# NeuroScope AI — Acquire qualifying disease datasets

Run actual transfers only in Google Colab with private Google Drive storage. Do not run downloads or training in Codex Cloud. Every disease switch defaults to **False**. No model training is included.

This continues the existing notebook and reuses the runtime, dependency and Drive setup from `Brain_MRI_Detection_Colab.ipynb` on `feature/detection-setup`. That original notebook and all segmentation files are unchanged. The helpers and pinned catalog are embedded; no executable helper is fetched at runtime.

Five sources are prepared for download, with different readiness levels. Glioma, PROTEAS brain metastasis and ISLES have publisher checksums and documented identity conventions. The preferred larger meningioma source needs a reviewed identity/diagnostic map after file inventory. OASIS-2 has an explicit weaker integrity policy and needs clinical-group mapping. Download completion never means approval for training.

The existing **578 IXI-T2 files** at `/content/drive/MyDrive/BrainMRI/datasets/IXI/Healthy/IXI-T2/` are not read, scanned, modified, moved or downloaded. Keep MRI data, clinical metadata and notebook outputs out of GitHub.


## Five-disease comparison, updated 2026-10-22

| Disease | Source prepared for download | Independent people / counts | Compressed download | Readiness / research terms |
|---|---|---|---|---|
| Glioma | MSD Task01, authorized MONAI S3 mirror | 750 case volumes; 484 annotated; clinical patient identity not independently proven | 7.61 GB | Publisher MD5; CC BY-SA 4.0; BraTS overlap |
| Meningioma | Preferred: Hwang/SNUBH, Zenodo 5945963 | 91 meningioma patients, 171 scans. Additional lesion-negative lung-cancer controls: body says 10, supplement says 12. | 20.86 GB | Publisher MD5; CC BY 4.0; file inventory then explicit patient/diagnosis mapping |
| Brain metastasis | PROTEAS, latest pinned Zenodo 23171182 | 40 patients; 185 studies; 65 metastases; 45 course archives | 15.13 GB including clinical XLSX | Publisher MD5; CC BY 4.0; a/b courses share patient identity |
| Alzheimer's | Official OASIS-2 legacy raw archives | 150 people: 64 AD at baseline, 72 stable controls, 14 converters; 373 sessions | 19.32 GB | Custom OASIS terms; **no publisher hash**; full gzip CRC32 + local SHA-256; clinical-group map required |
| Post-stroke | ISLES 2022 public training release | 250 annotated cases from three centers | 1.69 GB | Publisher MD5; CC BY 4.0; acute/subacute ischemic lesions |

The original OpenNeuro 37-person meningioma baseline subset remains selectable below (4.41 GB, CC0). It is an olfactory-groove-only, single-center cohort, not representative of all meningiomas. The larger SNUBH cohort covers more anatomical sites, but is still single-center and has repeat scans and unresolved control-count reporting.

OASIS terms are published by the project's maintainers at https://www.nitrc.org/projects/oasis/. The official NrgXnat README explicitly distinguishes direct OASIS-1/2 access from signed-DUA/project access for later releases. This notebook uses only the official Washington University legacy server, not third-party MRI copies. Local SHA-256 is a repeat-use integrity baseline, **not publisher-hash verification**. If publisher-hash verification is essential to your use, leave the Alzheimer's switch off.

The full report with source comparisons, citations, exact links, count semantics and unresolved issues is `detection/research/DATASET_SELECTION.md`.


## 1. Configure setup

Keep the default GPU requirement in Colab. For a lightweight CPU check, set
`REQUIRE_GPU = False`, `INSTALL_COLAB_DEPENDENCIES = False`, and `MOUNT_GOOGLE_DRIVE = False`.
Edit `PROJECT_ROOT` to your private project folder. Optional overrides can point to an existing dataset
and a future detection checkpoint. Missing data or weights do not prevent setup checks.


In [ ]:
from pathlib import Path

REQUIRE_GPU = True  # @param {type:"boolean"}
INSTALL_COLAB_DEPENDENCIES = True  # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
CREATE_OUTPUT_DIRS = False  # @param {type:"boolean"}

DRIVE_MOUNT_POINT = "/content/drive"  # @param {type:"string"}
PROJECT_ROOT = "/content/drive/MyDrive/BrainMRI"  # @param {type:"string"}
DATASET_ROOT_OVERRIDE = ""  # @param {type:"string"}
CHECKPOINT_PATH_OVERRIDE = ""  # @param {type:"string"}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Google Colab runtime:", IN_COLAB)


## 2. Check PyTorch, CUDA, and GPU

Use Colab's preinstalled CUDA-enabled PyTorch. This cell checks a tiny tensor operation on the selected
device; it does not create a model. If CUDA is unavailable, select a T4 runtime and reconnect.


In [ ]:
import sys
import torch

def check_torch_runtime(torch_module, require_gpu=True):
    cuda_available = torch_module.cuda.is_available()
    if require_gpu and not cuda_available:
        raise RuntimeError(
            "CUDA is unavailable. In Colab select Runtime > Change runtime type > "
            "T4 GPU and reconnect. Use REQUIRE_GPU=False only for CPU setup checks."
        )

    device = torch_module.device("cuda:0" if cuda_available else "cpu")
    with torch_module.no_grad():
        sample = torch_module.arange(4, dtype=torch_module.float32, device=device)
        if (sample * sample).sum().item() != 14.0:
            raise RuntimeError("PyTorch device arithmetic check failed.")
    if cuda_available:
        torch_module.cuda.synchronize()

    info = {
        "python": sys.version.split()[0],
        "pytorch": str(torch_module.__version__),
        "pytorch_cuda_build": torch_module.version.cuda,
        "cuda_available": cuda_available,
        "device": str(device),
        "gpu_name": torch_module.cuda.get_device_name(0) if cuda_available else None,
        "gpu_memory_gib": (
            round(torch_module.cuda.get_device_properties(0).total_memory / 2**30, 2)
            if cuda_available else None
        ),
        "device_tensor_check": "PASS",
    }
    return info

runtime_info = check_torch_runtime(torch, require_gpu=REQUIRE_GPU)
for key, value in runtime_info.items():
    print(f"{key}: {value}")
if runtime_info["gpu_name"] and "T4" not in runtime_info["gpu_name"]:
    print("This runtime has a different GPU; the project default is Tesla T4.")


## 3. Prepare MONAI and NiBabel

Only in Colab, install **MONAI 1.5.1** and **NiBabel 5.3.2**. PyTorch, CUDA, and NumPy remain supplied
by Colab; `--no-deps` prevents this step from replacing them. The next cell checks the installed
package requirements and runs small in-memory compatibility checks. If a requirement fails, stop
and use a compatible Colab runtime. If either library was already imported at a different version,
restart the session and rerun the notebook.


In [ ]:
import importlib.metadata as metadata
import subprocess

COLAB_PACKAGES = {"monai": "1.5.1", "nibabel": "5.3.2"}

if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
    requested = []
    for package, expected_version in COLAB_PACKAGES.items():
        try:
            current_version = metadata.version(package)
        except metadata.PackageNotFoundError:
            current_version = None
        if current_version != expected_version:
            requested.append(f"{package}=={expected_version}")
    if requested:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check", "--no-input", "--no-deps", *requested,
        ])
    else:
        print("Pinned Colab packages are already installed.")
else:
    print("Dependency installation skipped; existing packages will be checked.")


In [ ]:
import numpy as np
import monai
import nibabel as nib
from packaging.requirements import Requirement
from monai.transforms import NormalizeIntensity

def check_declared_requirements(package_names):
    problems = []
    for package in package_names:
        for requirement_text in metadata.requires(package) or []:
            requirement = Requirement(requirement_text)
            if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
                continue
            try:
                installed_version = metadata.version(requirement.name)
            except metadata.PackageNotFoundError:
                problems.append(f"{package}: missing {requirement.name}")
                continue
            if requirement.specifier and not requirement.specifier.contains(installed_version):
                problems.append(
                    f"{package}: {requirement.name} {installed_version} "
                    f"does not satisfy {requirement.specifier}"
                )
    if problems:
        raise RuntimeError("Package compatibility check failed:\n" + "\n".join(problems))

check_declared_requirements(["torch", "monai", "nibabel"])
for package, module in [("monai", monai), ("nibabel", nib)]:
    if module.__version__ != metadata.version(package):
        raise RuntimeError(f"{package} was updated after import; restart the Colab session.")
    if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
        if module.__version__ != COLAB_PACKAGES[package]:
            raise RuntimeError(f"Unexpected {package} version; restart and rerun setup.")

# Artificial arrays only; no MRI files or weights are read.
synthetic = np.arange(64, dtype=np.float32).reshape(4, 4, 4)
normalized = NormalizeIntensity()(synthetic)
normalized_tensor = torch.as_tensor(normalized)
assert tuple(normalized_tensor.shape) == (4, 4, 4)
assert torch.isfinite(normalized_tensor).all().item()
assert abs(normalized_tensor.mean().item()) < 1e-5

nifti = nib.Nifti1Image(synthetic, np.eye(4))
assert nifti.shape == (4, 4, 4)
np.testing.assert_array_equal(np.asarray(nifti.dataobj), synthetic)

compatibility_info = {
    "numpy": np.__version__,
    "monai": monai.__version__,
    "nibabel": nib.__version__,
    "declared_requirements": "PASS",
    "monai_transform": "PASS",
    "nifti_in_memory": "PASS",
}
for key, value in compatibility_info.items():
    print(f"{key}: {value}")


## 4. Optionally mount Google Drive

Mounting requests access through Colab's own authorization prompt. Do not put credentials in this
notebook. To work without Drive, disable mounting and change `PROJECT_ROOT` to a local Colab path
such as `/content/brain-mri-detection`. Local Colab files are temporary.


In [ ]:
DRIVE_MOUNTED = False
if MOUNT_GOOGLE_DRIVE:
    if not IN_COLAB:
        print("Drive mounting skipped: this is not a Google Colab runtime.")
    else:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT, force_remount=False)
        DRIVE_MOUNTED = (Path(DRIVE_MOUNT_POINT) / "MyDrive").is_dir()
        if not DRIVE_MOUNTED:
            raise RuntimeError("Google Drive MyDrive is unavailable after mounting.")
else:
    print("Google Drive mounting disabled.")


## Select diseases, meningioma source and storage budgets

All transfers go to private `Drive/datasets/DiseaseAcquisition/<dataset_id>/`. Different existing data are never overwritten. Archives remain on Drive, so reserve space for both compressed and extracted data. HTTP Range/If-Range enables resume where supported. Hwang transfers use configurable bounded chunks, connection/read timeouts, retries and progress/sync intervals below. Orphan resume metadata without partial bytes starts a fresh transfer explicitly; an actual partial without a trusted server validator is preserved and stops. A server that ignores Range never triggers an automatic destructive restart. See `detection/research/MENINGIOMA_INTERRUPTED_DOWNLOAD.md` for diagnosis and rerun instructions.

**Drive quota is not reliably reported by `shutil.disk_usage`.** Enter the free space shown in Google Drive as `DRIVE_AVAILABLE_GIB`. Zero blocks transfers. Mounted Drive filesystem capacity is not used as account quota. Real local Colab disk is checked separately with a 2 GiB runtime/cache reserve; MRI archives and extraction stream directly to Drive without local staging. `MAX_UNPACKED_GIB_PER_DISEASE` is one shared extraction budget across a disease's archives, not multiplied by PROTEAS's 45 course archives.

For the larger meningioma source and OASIS-2, the first run may end with `download_verified_inventory_pending_patient_map`. This is an explicit review state, not a successful clinical-label validation. Headers, hashes and known identities are inventoried, while missing identities/diagnoses stay null. A private `patient_map_template.csv` is generated. Use verified source metadata and review to fill a **separate** CSV, enter its Drive path below, then rerun. Never invent new patients from scan counts or infer a control label from an absent mask.

Mapping columns: `relative_path,patient_id,diagnosis,modality,role,visit_id`. Meningioma diagnoses are `meningioma` or `brain_lesion_negative_lung_cancer`; do not call these controls population-healthy. OASIS diagnoses are `AD_clinical_baseline`, `control_stable`, or `converter_excluded`. All visits of a converter must remain excluded from a binary AD/control training cohort. Missing information is not AD, MCI is not automatically AD, and generic CDR thresholds are not used to assign labels. Full reviewed OASIS mapping is checked against the published 64/72/14 patient groups.


In [ ]:
DOWNLOAD_GLIOMA = False  # @param {type:"boolean"}
DOWNLOAD_MENINGIOMA = False  # @param {type:"boolean"}
DOWNLOAD_BRAIN_METASTASIS = False  # @param {type:"boolean"}
DOWNLOAD_ALZHEIMERS = False  # @param {type:"boolean"}
DOWNLOAD_POST_STROKE = False  # @param {type:"boolean"}
MENINGIOMA_SOURCE = "hwang91"  # @param ["hwang91", "openneuro37"]
MENINGIOMA_REVIEWED_MAP_CSV = ""  # @param {type:"string"}
OASIS2_REVIEWED_MAP_CSV = ""  # @param {type:"string"}
DOWNLOAD_CONNECT_TIMEOUT_SECONDS = 30  # @param {type:"number"}
DOWNLOAD_READ_TIMEOUT_SECONDS = 60  # @param {type:"number"}
DOWNLOAD_MAX_RETRIES = 2  # @param {type:"integer"}
DOWNLOAD_CHUNK_MIB = 1  # @param {type:"integer"}
DOWNLOAD_PROGRESS_SECONDS = 10  # @param {type:"number"}
DOWNLOAD_SYNC_SECONDS = 5  # @param {type:"number"}
DOWNLOAD_OPTIONS = dict(connect_timeout=DOWNLOAD_CONNECT_TIMEOUT_SECONDS,
    read_timeout=DOWNLOAD_READ_TIMEOUT_SECONDS, max_retries=DOWNLOAD_MAX_RETRIES,
    chunk_bytes=int(DOWNLOAD_CHUNK_MIB * 2**20), progress_interval=DOWNLOAD_PROGRESS_SECONDS,
    sync_interval=DOWNLOAD_SYNC_SECONDS)
DRIVE_AVAILABLE_GIB = 0.0  # @param {type:"number"}
MAX_DOWNLOAD_GIB_PER_DISEASE = 25.0  # @param {type:"number"}
MAX_UNPACKED_GIB_PER_DISEASE = 60.0  # @param {type:"number"}
ACQUISITION_ROOT = str(Path(PROJECT_ROOT) / "datasets" / "DiseaseAcquisition")
SWITCHES = {
    "glioma": DOWNLOAD_GLIOMA, "meningioma": DOWNLOAD_MENINGIOMA,
    "brain_metastasis": DOWNLOAD_BRAIN_METASTASIS, "alzheimers": DOWNLOAD_ALZHEIMERS,
    "post_stroke": DOWNLOAD_POST_STROKE,
}


In [ ]:
import json
CATALOG = json.loads('{"glioma": {"id": "msd_task01", "disease": "Glioma", "eligible": true, "selection": "Practical authorized direct-download choice; not a claim of largest available cohort", "source": "https://medicaldecathlon.com/", "authorized_mirror_evidence": "https://raw.githubusercontent.com/Project-MONAI/MONAI/1.5.1/monai/apps/datasets.py", "license": "CC-BY-SA-4.0", "license_url": "https://creativecommons.org/licenses/by-sa/4.0/", "citation": "Antonelli et al. The Medical Segmentation Decathlon. Nature Communications 13, 4128 (2022). doi:10.1038/s41467-022-30695-9; cite BraTS source publications too.", "independent_patients": null, "expected_groups": 750, "labeled_cases": 484, "unlabeled_cases": 266, "count_note": "750 4D case volumes, not 3,000 patients. Unique clinical patient identity not independently established. BraTS 2016/17 overlap; do not pool as independent cohorts.", "download_probe": "200 HEAD, Content-Length 7608266240, Accept-Ranges bytes; official MONAI 1.5.1 MD5 verified; payload not fetched.", "assets": [{"path": "Task01_BrainTumour.tar", "url": "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar", "bytes": 7608266240, "checksum": "md5:240a19d752f0d9e9101544901065d872", "archive": true}], "limitations": ["7.61 GB archive (7.09 GiB), confirmed by HEAD; extraction needs additional storage. The previous notebook 2 GB estimate is incorrect.", "Training masks only for 484 cases; not all 750.", "Already represented in the segmentation project: reuse any existing verified archive in this acquisition destination; do not duplicate under BraTS names.", "License must also be confirmed from extracted dataset.json; no clinical approval is implied."], "download_bytes": 7608266240, "expected_min_masks": 484, "license_evidence": "https://medicaldecathlon.com/"}, "meningioma": {"id": "hwang_meningioma_5945963", "disease": "Meningioma", "eligible": true, "selection": "Preferred larger anatomically broader author-released 3D cohort; original 37-patient choice retained as an alternative", "source": "https://zenodo.org/records/5945963", "metadata_source": "https://zenodo.org/api/records/5945963", "paper_source": "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC9782766/fullTextXML", "license": "CC-BY-4.0", "license_url": "https://creativecommons.org/licenses/by/4.0/", "citation": "Hwang et al. Fully Automated Segmentation Models of Supratentorial Meningiomas Assisted by Inclusion of Normal Brain Images. Journal of Imaging 8(12), 327 (2022). doi:10.3390/jimaging8120327; dataset doi:10.5281/zenodo.5945963", "independent_patients": 91, "count_note": "91 meningioma patients contribute 171 scans: 115 scans from 35 followed patients plus 56 single-scan patients. Additional lesion-negative lung-cancer subjects: paper body says 10, supplementary Table S1 says 12; total/control counts are unresolved.", "modalities": ["T1w", "T1ce (3D, 1 mm acquisition)", "T2w", "FLAIR"], "formats": ["NIfTI .nii / .nii.gz"], "download_bytes": 20855300096, "download_probe": "Official author-deposited open Zenodo release and direct HEAD 200 size confirmed; publisher MD5 supplied. MRI archive not downloaded or listed in Cloud.", "assets": [{"path": "Nii_data.tar.gz", "url": "https://zenodo.org/api/records/5945963/files/Nii_data.tar.gz/content", "bytes": 20855300096, "checksum": "md5:c3b2ce5fbad44e1885ac73a03be9d38b", "archive": true}], "expected_diagnosis_groups": {"meningioma": 91}, "patient_map_required": true, "limitations": ["One tertiary hospital; 48/91 convexity tumors, 22/91 skull-base attachment, only 3/91 olfactory-groove cases. Broader than ds007345, not population representative.", "Published clinical protocol includes four sequences; the exact archive channel/mask inventory must be inspected in Colab. Do not assume every acquired sequence or mask is released.", "Paper body uses 10 lesion-negative controls; supplement Table S1 lists 12. These subjects had lung cancer, not population-healthy IXI-like controls.", "Published archive filenames/longitudinal ID mapping were not independently established without reading MRI data. Download first into private Drive, inspect headers, then provide an explicit reviewed file-to-patient/diagnosis/visit CSV. No scan IDs are invented as patient IDs.", "With no reviewed map the notebook produces a file inventory and a blank mapping template, reports download verified / mapping pending, and does not claim a verified patient-level manifest or training readiness.", "The article data-availability statement has a 10.5271 DOI typo; use verified Zenodo DOI prefix 10.5281."], "alternatives": {"openneuro37": {"id": "openneuro_ds007345", "disease": "Meningioma", "eligible": true, "selection": "Verified ungated structural-MRI alternative to larger controlled-access releases", "source": "https://openneuro.org/datasets/ds007345/versions/1.0.0", "metadata_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/dataset_description.json", "license": "CC0-1.0", "license_url": "https://creativecommons.org/publicdomain/zero/1.0/", "citation": "Filimonova et al. Multimodal Brain MRI and Clinical Data in Olfactory Groove Meningioma: A Prospective Data Report. doi:10.18112/openneuro.ds007345.v1.0.0", "independent_patients": 37, "expected_groups": 37, "release": "1.0.0", "metadata_commit": "fd052d49bfa51677fca817efdb68f237eeda9774", "download_bytes": 4407701072, "download_probe": "All 179 selected S3 assets returned HEAD 200 with matching byte sizes and Range support. Every publisher SHA256E pointer was verified. No MRI payload fetched in Cloud.", "assets": [{"path": "derivatives/sub-olf01/segmentation/sub-olf01_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf01/segmentation/sub-olf01_tumor_mask.nii.gz", "bytes": 105697, "checksum": "sha256:7b7715489e4e2e14f835ace3f1ab17490a08c5d9b05b78b901c376bfefed6910", "patient_id": "sub-olf01", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf01/segmentation/sub-olf01_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf02/segmentation/sub-olf02_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf02/segmentation/sub-olf02_edema_mask.nii.gz", "bytes": 295013, "checksum": "sha256:eb9a4729f85002f99c007357a7b6a6c55d4742ade059ad68773cd34a91bf7207", "patient_id": "sub-olf02", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf02/segmentation/sub-olf02_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf02/segmentation/sub-olf02_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf02/segmentation/sub-olf02_tumor_mask.nii.gz", "bytes": 111809, "checksum": "sha256:4450f6e942b126d7a1da2dd519c9f2e0c02b643db0245de6861aab281d994125", "patient_id": "sub-olf02", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf02/segmentation/sub-olf02_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf03/segmentation/sub-olf03_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf03/segmentation/sub-olf03_edema_mask.nii.gz", "bytes": 201309, "checksum": "sha256:a1d72d7d547d6fe718fb7602c657e18bcd2db56e19d29760071a847ee30c119e", "patient_id": "sub-olf03", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf03/segmentation/sub-olf03_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf03/segmentation/sub-olf03_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf03/segmentation/sub-olf03_tumor_mask.nii.gz", "bytes": 107366, "checksum": "sha256:98737dc16e31186d11f5deb25bdef332cca78f8e5768a862de9277c01c2812ce", "patient_id": "sub-olf03", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf03/segmentation/sub-olf03_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf04/segmentation/sub-olf04_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf04/segmentation/sub-olf04_edema_mask.nii.gz", "bytes": 236203, "checksum": "sha256:c8ee8c006fe0e3cb8373a3ad00cee6eaff77933cf2130e6691b2e0930be0dd08", "patient_id": "sub-olf04", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf04/segmentation/sub-olf04_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf04/segmentation/sub-olf04_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf04/segmentation/sub-olf04_tumor_mask.nii.gz", "bytes": 112985, "checksum": "sha256:97f52f718a777355733928ca04304e527fa338b8b5744ed413eb1f2162aedbf7", "patient_id": "sub-olf04", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf04/segmentation/sub-olf04_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf05/segmentation/sub-olf05_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf05/segmentation/sub-olf05_edema_mask.nii.gz", "bytes": 218413, "checksum": "sha256:62cf87511e4db8c7f91527c165c868db9586bbd08b50678e0993d6ae59ac6c20", "patient_id": "sub-olf05", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf05/segmentation/sub-olf05_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf05/segmentation/sub-olf05_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf05/segmentation/sub-olf05_tumor_mask.nii.gz", "bytes": 105549, "checksum": "sha256:b2e7defff9617c79c253b474edd6f97555101d638090b6f8a424afdac25d7321", "patient_id": "sub-olf05", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf05/segmentation/sub-olf05_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf06/segmentation/sub-olf06_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf06/segmentation/sub-olf06_edema_mask.nii.gz", "bytes": 260088, "checksum": "sha256:83f76e9dd2adb8aa0f25ea64e650194a8d60bbf1dbf0462dd2f594d689ed12d5", "patient_id": "sub-olf06", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf06/segmentation/sub-olf06_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf06/segmentation/sub-olf06_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf06/segmentation/sub-olf06_tumor_mask.nii.gz", "bytes": 118913, "checksum": "sha256:c22ab68f8ba0f454ca7e34f00803584812a0c07769593767f9f3c75fe0ab14aa", "patient_id": "sub-olf06", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf06/segmentation/sub-olf06_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf07/segmentation/sub-olf07_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf07/segmentation/sub-olf07_edema_mask.nii.gz", "bytes": 300002, "checksum": "sha256:0ecd230d74c387ac4ada031fab432b523c76fcb45adae5bd018eb3be0c14a6f0", "patient_id": "sub-olf07", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf07/segmentation/sub-olf07_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf07/segmentation/sub-olf07_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf07/segmentation/sub-olf07_tumor_mask.nii.gz", "bytes": 105704, "checksum": "sha256:26f07920afd03291c867a47249802517a291014f39e22e87c2a8b596c2b0e7d8", "patient_id": "sub-olf07", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf07/segmentation/sub-olf07_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf08/segmentation/sub-olf08_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf08/segmentation/sub-olf08_edema_mask.nii.gz", "bytes": 233012, "checksum": "sha256:efe49c193759f2c36154e966a009cfa684428c6d5c2bc449e1a6be1f380e47ce", "patient_id": "sub-olf08", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf08/segmentation/sub-olf08_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf08/segmentation/sub-olf08_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf08/segmentation/sub-olf08_tumor_mask.nii.gz", "bytes": 124258, "checksum": "sha256:0ad8052df8d1601a88a3f8c446a0cd692ceed5ce4124071fa4b91707d8755764", "patient_id": "sub-olf08", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf08/segmentation/sub-olf08_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf09/segmentation/sub-olf09_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf09/segmentation/sub-olf09_tumor_mask.nii.gz", "bytes": 115778, "checksum": "sha256:315387ceca5786d748c5deb7a8186b4a4f75208b06f432aa5df2e6e8ea5c5a54", "patient_id": "sub-olf09", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf09/segmentation/sub-olf09_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf10/segmentation/sub-olf10_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf10/segmentation/sub-olf10_edema_mask.nii.gz", "bytes": 257317, "checksum": "sha256:018cfff517b27c2db446a76993fb8fd11c74046d31788bf1a40b8b8899f86f52", "patient_id": "sub-olf10", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf10/segmentation/sub-olf10_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf10/segmentation/sub-olf10_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf10/segmentation/sub-olf10_tumor_mask.nii.gz", "bytes": 107075, "checksum": "sha256:767dea0007283121900b34e6e744c9a5461a81c8a5997d21ed78f4fff12ecf80", "patient_id": "sub-olf10", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf10/segmentation/sub-olf10_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf11/segmentation/sub-olf11_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf11/segmentation/sub-olf11_edema_mask.nii.gz", "bytes": 191178, "checksum": "sha256:73698328ce68ea01dfc799853be2b7bbf595ef4d8cb2c2e80d4b04589b06812c", "patient_id": "sub-olf11", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf11/segmentation/sub-olf11_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf11/segmentation/sub-olf11_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf11/segmentation/sub-olf11_tumor_mask.nii.gz", "bytes": 109527, "checksum": "sha256:ee43795fafea7ee445e771c8d3b97a941d7541d8d6845760b83891a062049b77", "patient_id": "sub-olf11", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf11/segmentation/sub-olf11_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf12/segmentation/sub-olf12_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf12/segmentation/sub-olf12_edema_mask.nii.gz", "bytes": 265135, "checksum": "sha256:f3f0ddb62b10c42855b121cc76af959422204b56b5f5039f18311445710ce124", "patient_id": "sub-olf12", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf12/segmentation/sub-olf12_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf12/segmentation/sub-olf12_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf12/segmentation/sub-olf12_tumor_mask.nii.gz", "bytes": 109936, "checksum": "sha256:b32a0709844fdecf55f83b7c30943792c9695f98d9226143377ce113a7fef031", "patient_id": "sub-olf12", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf12/segmentation/sub-olf12_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf13/segmentation/sub-olf13_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf13/segmentation/sub-olf13_tumor_mask.nii.gz", "bytes": 102825, "checksum": "sha256:2defc7a16ddd4e5db6bf36b099b79ffd993d2be74a8ef64b001876568de1f041", "patient_id": "sub-olf13", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf13/segmentation/sub-olf13_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf14/segmentation/sub-olf14_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf14/segmentation/sub-olf14_edema_mask.nii.gz", "bytes": 202848, "checksum": "sha256:11a76022776a5f7341f82439446ff5ac2438c26149b8cf9b1f8a066788003ff7", "patient_id": "sub-olf14", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf14/segmentation/sub-olf14_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf14/segmentation/sub-olf14_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf14/segmentation/sub-olf14_tumor_mask.nii.gz", "bytes": 102052, "checksum": "sha256:ddf0bdd50204cab98af56141581a901e9427d51ab8c42d03ebbf48336f063f45", "patient_id": "sub-olf14", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf14/segmentation/sub-olf14_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf15/segmentation/sub-olf15_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf15/segmentation/sub-olf15_tumor_mask.nii.gz", "bytes": 122204, "checksum": "sha256:b6c5a4582aeb3f800c29098361ba5ad16e10ce63ea796876e5018ea8fe99af89", "patient_id": "sub-olf15", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf15/segmentation/sub-olf15_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf16/segmentation/sub-olf16_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf16/segmentation/sub-olf16_edema_mask.nii.gz", "bytes": 195727, "checksum": "sha256:16374699cd5b6f916528ccd002a20abfa107be4ab700572b9a85139bbef61cf1", "patient_id": "sub-olf16", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf16/segmentation/sub-olf16_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf16/segmentation/sub-olf16_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf16/segmentation/sub-olf16_tumor_mask.nii.gz", "bytes": 105007, "checksum": "sha256:2eb55f9e707e60f26f24143bdacdf10169a379410ac891b1334517aef23b8c64", "patient_id": "sub-olf16", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf16/segmentation/sub-olf16_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf17/segmentation/sub-olf17_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf17/segmentation/sub-olf17_edema_mask.nii.gz", "bytes": 283446, "checksum": "sha256:0078b418e62a8489540b86746d1fed722b4a19040c87fee84196ed5bfac881a3", "patient_id": "sub-olf17", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf17/segmentation/sub-olf17_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf17/segmentation/sub-olf17_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf17/segmentation/sub-olf17_tumor_mask.nii.gz", "bytes": 110416, "checksum": "sha256:1ef8ca06990f07806b384294438e17e91948394153be2a70588314c0d8619fe1", "patient_id": "sub-olf17", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf17/segmentation/sub-olf17_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf18/segmentation/sub-olf18_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf18/segmentation/sub-olf18_edema_mask.nii.gz", "bytes": 230844, "checksum": "sha256:8e96a17d42f1835e168853bdec1663c713a089f3d756370362db29a981cfbb90", "patient_id": "sub-olf18", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf18/segmentation/sub-olf18_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf18/segmentation/sub-olf18_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf18/segmentation/sub-olf18_tumor_mask.nii.gz", "bytes": 119233, "checksum": "sha256:75167b60c252bd971ba32c96afb57809986e7dbecc11ee7c0d98f346263e8c31", "patient_id": "sub-olf18", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf18/segmentation/sub-olf18_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf19/segmentation/sub-olf19_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf19/segmentation/sub-olf19_edema_mask.nii.gz", "bytes": 29542, "checksum": "sha256:bf3cce67d7a283eb08d710cb1e95b0315d093ac9649679d5e6f794e222fe635b", "patient_id": "sub-olf19", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf19/segmentation/sub-olf19_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf19/segmentation/sub-olf19_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf19/segmentation/sub-olf19_tumor_mask.nii.gz", "bytes": 143211, "checksum": "sha256:061b3d683c0068b649933868386c1bbfa76bc70f5e347f80b224a81cced6d04d", "patient_id": "sub-olf19", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf19/segmentation/sub-olf19_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf20/segmentation/sub-olf20_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf20/segmentation/sub-olf20_edema_mask.nii.gz", "bytes": 282350, "checksum": "sha256:03203f50ec0d57d1178212a47db79b4fd0fd0e71c2d36189a27b01e5583e6807", "patient_id": "sub-olf20", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf20/segmentation/sub-olf20_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf20/segmentation/sub-olf20_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf20/segmentation/sub-olf20_tumor_mask.nii.gz", "bytes": 114631, "checksum": "sha256:5bc21c27aaa780f9570d0a11ed9e307452200a639d3083d0ff33375e6d4653d3", "patient_id": "sub-olf20", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf20/segmentation/sub-olf20_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf21/segmentation/sub-olf21_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf21/segmentation/sub-olf21_edema_mask.nii.gz", "bytes": 352158, "checksum": "sha256:91ac73b95b3f76295b5318a76fd6a19eee316b7417b7d17f01e15190a0cc3f2c", "patient_id": "sub-olf21", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf21/segmentation/sub-olf21_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf21/segmentation/sub-olf21_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf21/segmentation/sub-olf21_tumor_mask.nii.gz", "bytes": 100373, "checksum": "sha256:a60aa71b3a746f7d8dd53b1df18c1c426da6dc9c0edb609a60e8532f9a79c4a3", "patient_id": "sub-olf21", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf21/segmentation/sub-olf21_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf22/segmentation/sub-olf22_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf22/segmentation/sub-olf22_edema_mask.nii.gz", "bytes": 307415, "checksum": "sha256:f95817b52eac943c6ccacd869ea5116bd1984795f53703605481c93ee19bdca5", "patient_id": "sub-olf22", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf22/segmentation/sub-olf22_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf22/segmentation/sub-olf22_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf22/segmentation/sub-olf22_tumor_mask.nii.gz", "bytes": 110879, "checksum": "sha256:3291adc0176bd5aa58496bbcd6ce0c664b7f3669ed854f25430ce32abdb90de6", "patient_id": "sub-olf22", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf22/segmentation/sub-olf22_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf23/segmentation/sub-olf23_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf23/segmentation/sub-olf23_edema_mask.nii.gz", "bytes": 222059, "checksum": "sha256:03f075ec990f0eed566ee93553e9951fdf0142bddda3f53ad946ab5d87a55b14", "patient_id": "sub-olf23", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf23/segmentation/sub-olf23_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf23/segmentation/sub-olf23_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf23/segmentation/sub-olf23_tumor_mask.nii.gz", "bytes": 105382, "checksum": "sha256:b6ca733cf84cec84e52dca35efc5e449b69e3451e45f2e2ac36a373dad5e69d6", "patient_id": "sub-olf23", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf23/segmentation/sub-olf23_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf24/segmentation/sub-olf24_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf24/segmentation/sub-olf24_edema_mask.nii.gz", "bytes": 191627, "checksum": "sha256:361f73154861d398698aaddbbb757aeb0d14ae3cf48af5244e50a439139f6258", "patient_id": "sub-olf24", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf24/segmentation/sub-olf24_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf24/segmentation/sub-olf24_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf24/segmentation/sub-olf24_tumor_mask.nii.gz", "bytes": 125947, "checksum": "sha256:75560e39e838ad7e4a9106b00b686b32922717b4ce47cc5e3c3a5a42171e1e2c", "patient_id": "sub-olf24", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf24/segmentation/sub-olf24_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf25/segmentation/sub-olf25_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf25/segmentation/sub-olf25_edema_mask.nii.gz", "bytes": 244715, "checksum": "sha256:29672f2e9ad6c573794be6ae474060f3e6923648b477e117faeff2a61a4dc464", "patient_id": "sub-olf25", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf25/segmentation/sub-olf25_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf25/segmentation/sub-olf25_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf25/segmentation/sub-olf25_tumor_mask.nii.gz", "bytes": 107978, "checksum": "sha256:6fdc221d526ff0277ffe6ff4abee3fb4caef37b4ae1ed02eb5d826d6860f74ef", "patient_id": "sub-olf25", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf25/segmentation/sub-olf25_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf26/segmentation/sub-olf26_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf26/segmentation/sub-olf26_edema_mask.nii.gz", "bytes": 191910, "checksum": "sha256:6cc3e5343132f0d3aa4a83cc31516ef437d241fe9367a436557e1c733d680a97", "patient_id": "sub-olf26", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf26/segmentation/sub-olf26_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf26/segmentation/sub-olf26_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf26/segmentation/sub-olf26_tumor_mask.nii.gz", "bytes": 103688, "checksum": "sha256:e58a1d41a7f2889ff25170ada6d35c5b61b2c61733b8d325a249a875ee27e537", "patient_id": "sub-olf26", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf26/segmentation/sub-olf26_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf27/segmentation/sub-olf27_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf27/segmentation/sub-olf27_edema_mask.nii.gz", "bytes": 272120, "checksum": "sha256:cee39ac1e0539da511e8f950f1c466cd6af295a3010aa464fefe8cbfd3848f79", "patient_id": "sub-olf27", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf27/segmentation/sub-olf27_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf27/segmentation/sub-olf27_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf27/segmentation/sub-olf27_tumor_mask.nii.gz", "bytes": 111513, "checksum": "sha256:67845a9c304d5bb94e10f50963ed3665232865c6faab1897870b735e477d5825", "patient_id": "sub-olf27", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf27/segmentation/sub-olf27_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf28/segmentation/sub-olf28_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf28/segmentation/sub-olf28_edema_mask.nii.gz", "bytes": 258967, "checksum": "sha256:2e7eba8886a3d6e8dac38c630e8547ce3c007b5d37bbda95ae565c32fc898baf", "patient_id": "sub-olf28", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf28/segmentation/sub-olf28_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf28/segmentation/sub-olf28_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf28/segmentation/sub-olf28_tumor_mask.nii.gz", "bytes": 113882, "checksum": "sha256:ebbb65f9eeb998dbf41774b114eda58d4aefd3baf4e61b2da641654aaeb99989", "patient_id": "sub-olf28", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf28/segmentation/sub-olf28_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf29/segmentation/sub-olf29_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf29/segmentation/sub-olf29_edema_mask.nii.gz", "bytes": 219764, "checksum": "sha256:be3655ecf708cfc0a02de99ecf6a23e18ef946e0d2571aaaff3823bad7916bf0", "patient_id": "sub-olf29", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf29/segmentation/sub-olf29_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf29/segmentation/sub-olf29_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf29/segmentation/sub-olf29_tumor_mask.nii.gz", "bytes": 104170, "checksum": "sha256:baa55747690494314e66b4c17524c4e2f3422d4b52f28d582d4fad1cfc6ceadd", "patient_id": "sub-olf29", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf29/segmentation/sub-olf29_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf30/segmentation/sub-olf30_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf30/segmentation/sub-olf30_edema_mask.nii.gz", "bytes": 224352, "checksum": "sha256:896b602c5e38be45c5be2ae23a51892cbf7f3ff275a87d11df1868af61524bbd", "patient_id": "sub-olf30", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf30/segmentation/sub-olf30_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf30/segmentation/sub-olf30_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf30/segmentation/sub-olf30_tumor_mask.nii.gz", "bytes": 124513, "checksum": "sha256:6bbb9890ae61f7b8257964f9518a35339478297a232a4e988b2638e4cb571405", "patient_id": "sub-olf30", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf30/segmentation/sub-olf30_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf31/segmentation/sub-olf31_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf31/segmentation/sub-olf31_edema_mask.nii.gz", "bytes": 264377, "checksum": "sha256:4f7e0f5233a2d6314be9d3745fd66c7465b3942b09228eeecdb7cebcf1076394", "patient_id": "sub-olf31", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf31/segmentation/sub-olf31_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf31/segmentation/sub-olf31_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf31/segmentation/sub-olf31_tumor_mask.nii.gz", "bytes": 113213, "checksum": "sha256:c85c9d0232d9eeadf51296bf775704aa9b62d01787ca0c833944c7d4c9552fdd", "patient_id": "sub-olf31", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf31/segmentation/sub-olf31_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf32/segmentation/sub-olf32_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf32/segmentation/sub-olf32_edema_mask.nii.gz", "bytes": 252579, "checksum": "sha256:e7785a6f0bc961b657e8a10c065fbf7fc779b1588308d7c60b6208aee95c4360", "patient_id": "sub-olf32", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf32/segmentation/sub-olf32_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf32/segmentation/sub-olf32_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf32/segmentation/sub-olf32_tumor_mask.nii.gz", "bytes": 123734, "checksum": "sha256:c6825844502f7f0f09185a3a52d944d74217f91e33966b04672868905844caff", "patient_id": "sub-olf32", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf32/segmentation/sub-olf32_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf33/segmentation/sub-olf33_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf33/segmentation/sub-olf33_tumor_mask.nii.gz", "bytes": 125197, "checksum": "sha256:0197aec088b91466614703e8c2e5b42ccfbda9d98d2717fcc83e2432305f8299", "patient_id": "sub-olf33", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf33/segmentation/sub-olf33_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf34/segmentation/sub-olf34_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf34/segmentation/sub-olf34_tumor_mask.nii.gz", "bytes": 125131, "checksum": "sha256:116894e0cdffef2804617e7076e5920cf31c2084dd93df64d51a8bf0284b4d5e", "patient_id": "sub-olf34", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf34/segmentation/sub-olf34_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf35/segmentation/sub-olf35_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf35/segmentation/sub-olf35_edema_mask.nii.gz", "bytes": 336594, "checksum": "sha256:f0c5178d754051660ebcb439240ea266efbdcbc3737152d92f092e3874dfaa74", "patient_id": "sub-olf35", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf35/segmentation/sub-olf35_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf35/segmentation/sub-olf35_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf35/segmentation/sub-olf35_tumor_mask.nii.gz", "bytes": 107420, "checksum": "sha256:77d96b72ff1147044159be3a17608f9f12ae32e15eb722afc4554f1c1e4935e6", "patient_id": "sub-olf35", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf35/segmentation/sub-olf35_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf36/segmentation/sub-olf36_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf36/segmentation/sub-olf36_edema_mask.nii.gz", "bytes": 225976, "checksum": "sha256:4fea2086e58e53aaa737c442dbe17b09331bb729cd76660d9a31823c80442c5e", "patient_id": "sub-olf36", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf36/segmentation/sub-olf36_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf36/segmentation/sub-olf36_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf36/segmentation/sub-olf36_tumor_mask.nii.gz", "bytes": 121547, "checksum": "sha256:f6078fcb65c2f9d7b29f5c4c3d5900583e7983b3d4b0d5679b81252b7fc0c74b", "patient_id": "sub-olf36", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf36/segmentation/sub-olf36_tumor_mask.nii.gz"}, {"path": "derivatives/sub-olf37/segmentation/sub-olf37_edema_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf37/segmentation/sub-olf37_edema_mask.nii.gz", "bytes": 267171, "checksum": "sha256:9b821b1a9b0c00535d1faebe58c503c649a8e4b5ac1a30d46ba175df0bc47c40", "patient_id": "sub-olf37", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf37/segmentation/sub-olf37_edema_mask.nii.gz"}, {"path": "derivatives/sub-olf37/segmentation/sub-olf37_tumor_mask.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/derivatives/sub-olf37/segmentation/sub-olf37_tumor_mask.nii.gz", "bytes": 105137, "checksum": "sha256:6d9dae8dedf632e9d6da03c64cd86ae7b08c42eb4c807f26dc26afc24f560e4c", "patient_id": "sub-olf37", "role": "mask", "modality": "mask", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/derivatives/sub-olf37/segmentation/sub-olf37_tumor_mask.nii.gz"}, {"path": "sub-olf01/anat/sub-olf01_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf01/anat/sub-olf01_T2w.nii.gz", "bytes": 52046803, "checksum": "sha256:f61430a825045f5d7a4011a16e5982a3ef59745fa169b5ad3b6afafb07b03c0f", "patient_id": "sub-olf01", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf01/anat/sub-olf01_T2w.nii.gz"}, {"path": "sub-olf01/anat/sub-olf01_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf01/anat/sub-olf01_run-01_T1w.nii.gz", "bytes": 26564886, "checksum": "sha256:32fab159e3f8273a58448740fecd6c36e85c83c8a2bf6183982678d4b72cadf7", "patient_id": "sub-olf01", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf01/anat/sub-olf01_run-01_T1w.nii.gz"}, {"path": "sub-olf01/anat/sub-olf01_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf01/anat/sub-olf01_run-02_T1w.nii.gz", "bytes": 29136633, "checksum": "sha256:16136a0c473cc44cdaab681131ea681e56384c99e7e826f30397c697c73e6e3d", "patient_id": "sub-olf01", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf01/anat/sub-olf01_run-02_T1w.nii.gz"}, {"path": "sub-olf02/anat/sub-olf02_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf02/anat/sub-olf02_T2w.nii.gz", "bytes": 54703793, "checksum": "sha256:df0b1beb7922e3862d2ad73f07172e1ce8836978b949e421410d260e0564a268", "patient_id": "sub-olf02", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf02/anat/sub-olf02_T2w.nii.gz"}, {"path": "sub-olf02/anat/sub-olf02_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf02/anat/sub-olf02_run-01_T1w.nii.gz", "bytes": 26493380, "checksum": "sha256:a0b7a01f9aa80712ee141afad2176cc279bc3abf0da1bb777c62f26827197a35", "patient_id": "sub-olf02", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf02/anat/sub-olf02_run-01_T1w.nii.gz"}, {"path": "sub-olf02/anat/sub-olf02_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf02/anat/sub-olf02_run-02_T1w.nii.gz", "bytes": 32703957, "checksum": "sha256:627881f4876ff2a39460dc74396315482fd24a50ad363947fe932754b8b8533a", "patient_id": "sub-olf02", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf02/anat/sub-olf02_run-02_T1w.nii.gz"}, {"path": "sub-olf03/anat/sub-olf03_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf03/anat/sub-olf03_T2w.nii.gz", "bytes": 49290364, "checksum": "sha256:c4a77201c7795a5beca141a1a007c4e92b18b7949961723cf139d21c0d30a204", "patient_id": "sub-olf03", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf03/anat/sub-olf03_T2w.nii.gz"}, {"path": "sub-olf03/anat/sub-olf03_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf03/anat/sub-olf03_run-01_T1w.nii.gz", "bytes": 26434171, "checksum": "sha256:61826fb78334ac14a566809d364967dd06cee43a0321994c9d06ccd57018cad0", "patient_id": "sub-olf03", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf03/anat/sub-olf03_run-01_T1w.nii.gz"}, {"path": "sub-olf03/anat/sub-olf03_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf03/anat/sub-olf03_run-02_T1w.nii.gz", "bytes": 28519099, "checksum": "sha256:9aec6ca7b7f0d183ef42d912a9be40f67702549c953767fbacfe1cd4b4925967", "patient_id": "sub-olf03", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf03/anat/sub-olf03_run-02_T1w.nii.gz"}, {"path": "sub-olf04/anat/sub-olf04_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf04/anat/sub-olf04_T2w.nii.gz", "bytes": 54644197, "checksum": "sha256:17864eb8585dc304d977384f5d5ade448af1d1a53f80eefc649c7aa34c296ba3", "patient_id": "sub-olf04", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf04/anat/sub-olf04_T2w.nii.gz"}, {"path": "sub-olf04/anat/sub-olf04_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf04/anat/sub-olf04_run-01_T1w.nii.gz", "bytes": 26863400, "checksum": "sha256:abef06fa7ebbd3bc8b60dfbd1fb642e537cd1b8128b3412eca7fb50324d73e43", "patient_id": "sub-olf04", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf04/anat/sub-olf04_run-01_T1w.nii.gz"}, {"path": "sub-olf04/anat/sub-olf04_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf04/anat/sub-olf04_run-02_T1w.nii.gz", "bytes": 28148902, "checksum": "sha256:b49eb2329d06fe4ae5f3f01fe6b2a3264a60d6cf9684bd29834b770a14b0c064", "patient_id": "sub-olf04", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf04/anat/sub-olf04_run-02_T1w.nii.gz"}, {"path": "sub-olf05/anat/sub-olf05_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf05/anat/sub-olf05_T2w.nii.gz", "bytes": 57538406, "checksum": "sha256:04e2f15ea51df49aadc680d1c5a2d9c975821963eece378b6966bc60568d1141", "patient_id": "sub-olf05", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf05/anat/sub-olf05_T2w.nii.gz"}, {"path": "sub-olf05/anat/sub-olf05_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf05/anat/sub-olf05_run-01_T1w.nii.gz", "bytes": 27166908, "checksum": "sha256:f888e924bb6fbab164bd2c3e5e95d303f03ce13995a34bda003d326e76382920", "patient_id": "sub-olf05", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf05/anat/sub-olf05_run-01_T1w.nii.gz"}, {"path": "sub-olf05/anat/sub-olf05_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf05/anat/sub-olf05_run-02_T1w.nii.gz", "bytes": 29808174, "checksum": "sha256:5fd2ed6d0966a74d710738bb1612556a5943f304adb3def473a8563a542bab94", "patient_id": "sub-olf05", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf05/anat/sub-olf05_run-02_T1w.nii.gz"}, {"path": "sub-olf06/anat/sub-olf06_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf06/anat/sub-olf06_T2w.nii.gz", "bytes": 60424073, "checksum": "sha256:0e3178016846eb76bf746c8f606a0401222245288386f2dde7d854c11ba11c99", "patient_id": "sub-olf06", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf06/anat/sub-olf06_T2w.nii.gz"}, {"path": "sub-olf06/anat/sub-olf06_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf06/anat/sub-olf06_run-01_T1w.nii.gz", "bytes": 29533500, "checksum": "sha256:21a88dbff98726922a2a379d9bf6e360b37b9960ed2f8b1353d11654601f7391", "patient_id": "sub-olf06", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf06/anat/sub-olf06_run-01_T1w.nii.gz"}, {"path": "sub-olf06/anat/sub-olf06_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf06/anat/sub-olf06_run-02_T1w.nii.gz", "bytes": 34469078, "checksum": "sha256:8b44047473a4d6dada7bd606b18b32e98e771cc31315e8d8b33333506fce083b", "patient_id": "sub-olf06", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf06/anat/sub-olf06_run-02_T1w.nii.gz"}, {"path": "sub-olf07/anat/sub-olf07_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf07/anat/sub-olf07_T2w.nii.gz", "bytes": 53090990, "checksum": "sha256:a3b5d6a31438f3b4a74dd20e7e95dab667f547cdfc078d5b07ade26ce1869f8c", "patient_id": "sub-olf07", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf07/anat/sub-olf07_T2w.nii.gz"}, {"path": "sub-olf07/anat/sub-olf07_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf07/anat/sub-olf07_run-01_T1w.nii.gz", "bytes": 27203938, "checksum": "sha256:51fc16024823441838abf30ff9cd1f3589a0cef8eddd338841253d1712048326", "patient_id": "sub-olf07", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf07/anat/sub-olf07_run-01_T1w.nii.gz"}, {"path": "sub-olf07/anat/sub-olf07_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf07/anat/sub-olf07_run-02_T1w.nii.gz", "bytes": 29164678, "checksum": "sha256:f78c80648cea18793f6a4f1229b31202872dd12ab65722c777c0d7a956ee424d", "patient_id": "sub-olf07", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf07/anat/sub-olf07_run-02_T1w.nii.gz"}, {"path": "sub-olf08/anat/sub-olf08_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf08/anat/sub-olf08_T2w.nii.gz", "bytes": 48387585, "checksum": "sha256:5b120d6ddf649120dcf82d8e5df2f9d0ab732eb44cb11611113dae59fb2c09a8", "patient_id": "sub-olf08", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf08/anat/sub-olf08_T2w.nii.gz"}, {"path": "sub-olf08/anat/sub-olf08_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf08/anat/sub-olf08_run-01_T1w.nii.gz", "bytes": 25366316, "checksum": "sha256:a12ebd36f95c3830dd48ee4532c33e64b2311c3b75bb137191d4bec5fa8e46c7", "patient_id": "sub-olf08", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf08/anat/sub-olf08_run-01_T1w.nii.gz"}, {"path": "sub-olf08/anat/sub-olf08_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf08/anat/sub-olf08_run-02_T1w.nii.gz", "bytes": 26557014, "checksum": "sha256:cb3677deab00eb8145d0f8a83b6f26fe591a271923d272a274f1749c2536c8de", "patient_id": "sub-olf08", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf08/anat/sub-olf08_run-02_T1w.nii.gz"}, {"path": "sub-olf09/anat/sub-olf09_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf09/anat/sub-olf09_T2w.nii.gz", "bytes": 59341057, "checksum": "sha256:431edf40769d87e1b0b2419d7796f074ebdfd365e1f088e28de9dbbd1d48afc0", "patient_id": "sub-olf09", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf09/anat/sub-olf09_T2w.nii.gz"}, {"path": "sub-olf09/anat/sub-olf09_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf09/anat/sub-olf09_run-01_T1w.nii.gz", "bytes": 27728670, "checksum": "sha256:2b1cacdc8a5dc4403a249a3a01add2e7813a85049713e3c529c284dff5d875a0", "patient_id": "sub-olf09", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf09/anat/sub-olf09_run-01_T1w.nii.gz"}, {"path": "sub-olf09/anat/sub-olf09_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf09/anat/sub-olf09_run-02_T1w.nii.gz", "bytes": 32637549, "checksum": "sha256:372627e5f6ef795fdfc4acad6294641b913104b791030501a46a2c82bbde4e82", "patient_id": "sub-olf09", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf09/anat/sub-olf09_run-02_T1w.nii.gz"}, {"path": "sub-olf10/anat/sub-olf10_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf10/anat/sub-olf10_T2w.nii.gz", "bytes": 54746691, "checksum": "sha256:bf4b014e7a94c98480be15fe82c1f6fcdfa201a4a8b3994d0803448ba3be9fe7", "patient_id": "sub-olf10", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf10/anat/sub-olf10_T2w.nii.gz"}, {"path": "sub-olf10/anat/sub-olf10_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf10/anat/sub-olf10_run-01_T1w.nii.gz", "bytes": 25601712, "checksum": "sha256:20c1377df3849aaf6fca8a0e81e67c27ce09a85fb34c513b43cd095eca62c52c", "patient_id": "sub-olf10", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf10/anat/sub-olf10_run-01_T1w.nii.gz"}, {"path": "sub-olf10/anat/sub-olf10_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf10/anat/sub-olf10_run-02_T1w.nii.gz", "bytes": 30659867, "checksum": "sha256:036a84398ada37c2c6523dec3cf01fa16c0c7ace38b751ea26601a80e6b10680", "patient_id": "sub-olf10", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf10/anat/sub-olf10_run-02_T1w.nii.gz"}, {"path": "sub-olf11/anat/sub-olf11_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf11/anat/sub-olf11_T2w.nii.gz", "bytes": 63288734, "checksum": "sha256:523482d4d4a8a276cce0871464d58a5cf5649ae11abaf8d8fdc03badbc9a1390", "patient_id": "sub-olf11", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf11/anat/sub-olf11_T2w.nii.gz"}, {"path": "sub-olf11/anat/sub-olf11_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf11/anat/sub-olf11_run-01_T1w.nii.gz", "bytes": 29156485, "checksum": "sha256:0d311f59a35fbac3c7a151d9f5439cd1f4b4d10aeeef6a33e516f98fd3ca611b", "patient_id": "sub-olf11", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf11/anat/sub-olf11_run-01_T1w.nii.gz"}, {"path": "sub-olf11/anat/sub-olf11_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf11/anat/sub-olf11_run-02_T1w.nii.gz", "bytes": 32414647, "checksum": "sha256:c938d9415ed53f4288ce57c29094279d5609b16dbd9009bd59fd1d967aa5baf1", "patient_id": "sub-olf11", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf11/anat/sub-olf11_run-02_T1w.nii.gz"}, {"path": "sub-olf12/anat/sub-olf12_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf12/anat/sub-olf12_T2w.nii.gz", "bytes": 58224705, "checksum": "sha256:1dc2a0bc69f84bbd674f250ca383960f1128a9cfc93374dc9d617deb011d6214", "patient_id": "sub-olf12", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf12/anat/sub-olf12_T2w.nii.gz"}, {"path": "sub-olf12/anat/sub-olf12_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf12/anat/sub-olf12_run-01_T1w.nii.gz", "bytes": 28506363, "checksum": "sha256:f279ff8712dce1abe75fbca35a88151cd3f4672d8ca2aa0bac9bf13dd221a71b", "patient_id": "sub-olf12", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf12/anat/sub-olf12_run-01_T1w.nii.gz"}, {"path": "sub-olf12/anat/sub-olf12_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf12/anat/sub-olf12_run-02_T1w.nii.gz", "bytes": 32614169, "checksum": "sha256:adaea676d755f13f43bd8f093b57d6f083529d067cc8e9bbde6786895e021c75", "patient_id": "sub-olf12", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf12/anat/sub-olf12_run-02_T1w.nii.gz"}, {"path": "sub-olf13/anat/sub-olf13_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf13/anat/sub-olf13_T2w.nii.gz", "bytes": 44658524, "checksum": "sha256:b7c0d100ca20c6f1f09525e0368dc8a528709051e393aa57de06a26904e85609", "patient_id": "sub-olf13", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf13/anat/sub-olf13_T2w.nii.gz"}, {"path": "sub-olf13/anat/sub-olf13_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf13/anat/sub-olf13_run-01_T1w.nii.gz", "bytes": 26670719, "checksum": "sha256:dc18c1ea0aca12594200937e659c3052087fe67992a3cba2eccf0157732e6d47", "patient_id": "sub-olf13", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf13/anat/sub-olf13_run-01_T1w.nii.gz"}, {"path": "sub-olf13/anat/sub-olf13_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf13/anat/sub-olf13_run-02_T1w.nii.gz", "bytes": 27754246, "checksum": "sha256:dbd757fdd03b9d2b0f2a5e0c9e42aca6c0e7ed62505f4b376e3d28dd745829d0", "patient_id": "sub-olf13", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf13/anat/sub-olf13_run-02_T1w.nii.gz"}, {"path": "sub-olf14/anat/sub-olf14_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf14/anat/sub-olf14_T2w.nii.gz", "bytes": 64374470, "checksum": "sha256:ad693c88d7b80b7407ea9144521e610817e7bf8d82f83988ef08800a05f280ba", "patient_id": "sub-olf14", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf14/anat/sub-olf14_T2w.nii.gz"}, {"path": "sub-olf14/anat/sub-olf14_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf14/anat/sub-olf14_run-01_T1w.nii.gz", "bytes": 29631084, "checksum": "sha256:5ae6083fb0df3f45c09728025a918b0873362861b655cbc4c28bd560ffc30b63", "patient_id": "sub-olf14", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf14/anat/sub-olf14_run-01_T1w.nii.gz"}, {"path": "sub-olf14/anat/sub-olf14_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf14/anat/sub-olf14_run-02_T1w.nii.gz", "bytes": 33065901, "checksum": "sha256:1527947776fedad553eca8146133a2ac314c32b2e168c6294c8ca76d1da07d2f", "patient_id": "sub-olf14", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf14/anat/sub-olf14_run-02_T1w.nii.gz"}, {"path": "sub-olf15/anat/sub-olf15_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf15/anat/sub-olf15_T2w.nii.gz", "bytes": 65779723, "checksum": "sha256:c16055d4fd3fc4a0e020f4b38032f3a0487a482298c037cfe7f02f813f93b039", "patient_id": "sub-olf15", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf15/anat/sub-olf15_T2w.nii.gz"}, {"path": "sub-olf15/anat/sub-olf15_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf15/anat/sub-olf15_run-01_T1w.nii.gz", "bytes": 31277376, "checksum": "sha256:d0223505605123dc6230d01065bfeae2b509617b6f9670bc7a7d7707a961dc1f", "patient_id": "sub-olf15", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf15/anat/sub-olf15_run-01_T1w.nii.gz"}, {"path": "sub-olf15/anat/sub-olf15_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf15/anat/sub-olf15_run-02_T1w.nii.gz", "bytes": 34894590, "checksum": "sha256:91807ef0730f81d063fb0d8d01cd11824cfd41b62e68056b01b2a54abdbbb6d6", "patient_id": "sub-olf15", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf15/anat/sub-olf15_run-02_T1w.nii.gz"}, {"path": "sub-olf16/anat/sub-olf16_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf16/anat/sub-olf16_T2w.nii.gz", "bytes": 47107039, "checksum": "sha256:0cafb730404c8c4d6ce3cc84c663f305eaf9661c550005aebc5cf410d0435047", "patient_id": "sub-olf16", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf16/anat/sub-olf16_T2w.nii.gz"}, {"path": "sub-olf16/anat/sub-olf16_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf16/anat/sub-olf16_run-01_T1w.nii.gz", "bytes": 23320473, "checksum": "sha256:1bed3c17b6a1a595c87057ff9174098937123470438f2e420e04d9eaa7c49016", "patient_id": "sub-olf16", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf16/anat/sub-olf16_run-01_T1w.nii.gz"}, {"path": "sub-olf16/anat/sub-olf16_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf16/anat/sub-olf16_run-02_T1w.nii.gz", "bytes": 25777505, "checksum": "sha256:2dbccebe4aa2fd95ba516c236897143a5cbb455a787377b1fdb8d1683475c319", "patient_id": "sub-olf16", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf16/anat/sub-olf16_run-02_T1w.nii.gz"}, {"path": "sub-olf17/anat/sub-olf17_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf17/anat/sub-olf17_T2w.nii.gz", "bytes": 74713934, "checksum": "sha256:8456ac5b7c492ecf1e3a96b291d92cd9a763e555a726f8fba8e10c89daebe3c9", "patient_id": "sub-olf17", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf17/anat/sub-olf17_T2w.nii.gz"}, {"path": "sub-olf17/anat/sub-olf17_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf17/anat/sub-olf17_run-01_T1w.nii.gz", "bytes": 38947507, "checksum": "sha256:c38be2e9b04052464f8f1f8d11fd1a87baab5de5398f1cbcb5b9d9452ff47c7e", "patient_id": "sub-olf17", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf17/anat/sub-olf17_run-01_T1w.nii.gz"}, {"path": "sub-olf17/anat/sub-olf17_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf17/anat/sub-olf17_run-02_T1w.nii.gz", "bytes": 43991818, "checksum": "sha256:bbb83d82c17485bac9f0d7b29b6d9be514c8dceda86669be80b7193c4bd5b700", "patient_id": "sub-olf17", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf17/anat/sub-olf17_run-02_T1w.nii.gz"}, {"path": "sub-olf18/anat/sub-olf18_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf18/anat/sub-olf18_T2w.nii.gz", "bytes": 66516117, "checksum": "sha256:d2cb61b094206a217e70c9010969cc160d6c4b97d0a5d980db18ff434047f02e", "patient_id": "sub-olf18", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf18/anat/sub-olf18_T2w.nii.gz"}, {"path": "sub-olf18/anat/sub-olf18_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf18/anat/sub-olf18_run-01_T1w.nii.gz", "bytes": 30228691, "checksum": "sha256:943d1da61bc150e0833ce232dda022532aad2a58274ce2aef89a1c81131bcfb6", "patient_id": "sub-olf18", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf18/anat/sub-olf18_run-01_T1w.nii.gz"}, {"path": "sub-olf18/anat/sub-olf18_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf18/anat/sub-olf18_run-02_T1w.nii.gz", "bytes": 33477204, "checksum": "sha256:034600813f64703beb5f801e01e7eff5508413972ee115721114b0975f832926", "patient_id": "sub-olf18", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf18/anat/sub-olf18_run-02_T1w.nii.gz"}, {"path": "sub-olf19/anat/sub-olf19_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf19/anat/sub-olf19_T2w.nii.gz", "bytes": 68157662, "checksum": "sha256:6ddcfbb415449f1012a27266313bd0117739cb13953d58d98503a0076291ac1b", "patient_id": "sub-olf19", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf19/anat/sub-olf19_T2w.nii.gz"}, {"path": "sub-olf19/anat/sub-olf19_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf19/anat/sub-olf19_run-01_T1w.nii.gz", "bytes": 27970297, "checksum": "sha256:0c8b894d5a0d19e3ee62051b42133bcc9b2f4340a2a9e361b147b2e1804955cd", "patient_id": "sub-olf19", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf19/anat/sub-olf19_run-01_T1w.nii.gz"}, {"path": "sub-olf19/anat/sub-olf19_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf19/anat/sub-olf19_run-02_T1w.nii.gz", "bytes": 36646205, "checksum": "sha256:b286abf2474fd2ea4e5107a9325039288b17fe439d8d236656785d8edeea49a5", "patient_id": "sub-olf19", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf19/anat/sub-olf19_run-02_T1w.nii.gz"}, {"path": "sub-olf20/anat/sub-olf20_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf20/anat/sub-olf20_T2w.nii.gz", "bytes": 74078418, "checksum": "sha256:afd37a4d5ecd338c20fd71d2a7a3a10664326947a53b3363351af7e0a23d3b6e", "patient_id": "sub-olf20", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf20/anat/sub-olf20_T2w.nii.gz"}, {"path": "sub-olf20/anat/sub-olf20_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf20/anat/sub-olf20_run-01_T1w.nii.gz", "bytes": 27205472, "checksum": "sha256:cd954a494aa8d06b1e902ffbef7b240dea5a0de78b80afe34287d3af0ccc4baa", "patient_id": "sub-olf20", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf20/anat/sub-olf20_run-01_T1w.nii.gz"}, {"path": "sub-olf20/anat/sub-olf20_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf20/anat/sub-olf20_run-02_T1w.nii.gz", "bytes": 33004680, "checksum": "sha256:00095280041911bf6df06ef55b85ec6e75eb15ba5ace917c024067c9bcd9ae39", "patient_id": "sub-olf20", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf20/anat/sub-olf20_run-02_T1w.nii.gz"}, {"path": "sub-olf21/anat/sub-olf21_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf21/anat/sub-olf21_T2w.nii.gz", "bytes": 58922872, "checksum": "sha256:3e79b92246089e82939c0bc134be94755592581bce9b835db3c94248fd696fd7", "patient_id": "sub-olf21", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf21/anat/sub-olf21_T2w.nii.gz"}, {"path": "sub-olf21/anat/sub-olf21_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf21/anat/sub-olf21_run-01_T1w.nii.gz", "bytes": 28543534, "checksum": "sha256:7279107041249f0f3b2602b3f63b97a278febb23ab4e62210522c65dff73b314", "patient_id": "sub-olf21", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf21/anat/sub-olf21_run-01_T1w.nii.gz"}, {"path": "sub-olf21/anat/sub-olf21_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf21/anat/sub-olf21_run-02_T1w.nii.gz", "bytes": 38501693, "checksum": "sha256:d2da9f594d2b4b0e593b5c9bac62d39f7fbe2045cdc2766e80dddcbcaa6ee765", "patient_id": "sub-olf21", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf21/anat/sub-olf21_run-02_T1w.nii.gz"}, {"path": "sub-olf22/anat/sub-olf22_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf22/anat/sub-olf22_T2w.nii.gz", "bytes": 59689076, "checksum": "sha256:1920479f0d47b5762b0a07e5ae447ca4b2871c553e3a8940cd51db9e902911a9", "patient_id": "sub-olf22", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf22/anat/sub-olf22_T2w.nii.gz"}, {"path": "sub-olf22/anat/sub-olf22_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf22/anat/sub-olf22_run-01_T1w.nii.gz", "bytes": 29765962, "checksum": "sha256:0ed12169f0a6ce6f09b6c85b1190b1fb3015ceb4c136369fb7e7348d43d4714b", "patient_id": "sub-olf22", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf22/anat/sub-olf22_run-01_T1w.nii.gz"}, {"path": "sub-olf22/anat/sub-olf22_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf22/anat/sub-olf22_run-02_T1w.nii.gz", "bytes": 32434598, "checksum": "sha256:a0857925f45f479d8f5ea84be885f38e66f288f58ee181f86a0c08ab9196e47d", "patient_id": "sub-olf22", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf22/anat/sub-olf22_run-02_T1w.nii.gz"}, {"path": "sub-olf23/anat/sub-olf23_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf23/anat/sub-olf23_T2w.nii.gz", "bytes": 57832122, "checksum": "sha256:a0df84611b1c512d5e1fd117b00745a54763be7fccf3c9afd68dbcee4ae3e078", "patient_id": "sub-olf23", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf23/anat/sub-olf23_T2w.nii.gz"}, {"path": "sub-olf23/anat/sub-olf23_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf23/anat/sub-olf23_run-01_T1w.nii.gz", "bytes": 26992412, "checksum": "sha256:e7b3c1b2e619e09992557942501cfe662b672fecbb454591eb25ec9f1c87f52d", "patient_id": "sub-olf23", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf23/anat/sub-olf23_run-01_T1w.nii.gz"}, {"path": "sub-olf23/anat/sub-olf23_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf23/anat/sub-olf23_run-02_T1w.nii.gz", "bytes": 29205413, "checksum": "sha256:ef00ecf44d287be0af94ca9d841ecdb53e6521a29756d98c63c42c0f31fea06e", "patient_id": "sub-olf23", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf23/anat/sub-olf23_run-02_T1w.nii.gz"}, {"path": "sub-olf24/anat/sub-olf24_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf24/anat/sub-olf24_T2w.nii.gz", "bytes": 49715106, "checksum": "sha256:f208faae7cb67a58bd461c93830f78176a68f5d39d7d77f93fdd47fc60427ffb", "patient_id": "sub-olf24", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf24/anat/sub-olf24_T2w.nii.gz"}, {"path": "sub-olf24/anat/sub-olf24_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf24/anat/sub-olf24_run-01_T1w.nii.gz", "bytes": 27004134, "checksum": "sha256:ea292b390948a5267f9b61e0f09b7022fb4940a4af5e9514bd093716984454a1", "patient_id": "sub-olf24", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf24/anat/sub-olf24_run-01_T1w.nii.gz"}, {"path": "sub-olf24/anat/sub-olf24_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf24/anat/sub-olf24_run-02_T1w.nii.gz", "bytes": 30527119, "checksum": "sha256:77a06e1bf868e88fd4dfbf8b89c519150ee4901f5c6a62b5a1427d936f456829", "patient_id": "sub-olf24", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf24/anat/sub-olf24_run-02_T1w.nii.gz"}, {"path": "sub-olf25/anat/sub-olf25_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf25/anat/sub-olf25_T2w.nii.gz", "bytes": 68499569, "checksum": "sha256:9b989d967e04b1548b7f9c9df2903b9e998327a1661e8cc61e39c2c4c60f9a6d", "patient_id": "sub-olf25", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf25/anat/sub-olf25_T2w.nii.gz"}, {"path": "sub-olf25/anat/sub-olf25_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf25/anat/sub-olf25_run-01_T1w.nii.gz", "bytes": 31948601, "checksum": "sha256:002b9ef91781383aec8c258aeaf49d2383381ce37bd0e615533ae62282c5192d", "patient_id": "sub-olf25", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf25/anat/sub-olf25_run-01_T1w.nii.gz"}, {"path": "sub-olf25/anat/sub-olf25_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf25/anat/sub-olf25_run-02_T1w.nii.gz", "bytes": 34918965, "checksum": "sha256:b0ce10836512a25a88d4eb5d759c81e87d8fed5e6397b1c0bc278ddf9f1c0675", "patient_id": "sub-olf25", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf25/anat/sub-olf25_run-02_T1w.nii.gz"}, {"path": "sub-olf26/anat/sub-olf26_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf26/anat/sub-olf26_T2w.nii.gz", "bytes": 54535231, "checksum": "sha256:a96d10561391d635a98cfd0dadd788fe82703e6d629b5b433eb4641588f355e1", "patient_id": "sub-olf26", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf26/anat/sub-olf26_T2w.nii.gz"}, {"path": "sub-olf26/anat/sub-olf26_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf26/anat/sub-olf26_run-01_T1w.nii.gz", "bytes": 23790075, "checksum": "sha256:07b9d5d36d3a98c55633f91b5dfdc7f19b47f99ec64c3a265f7be7ca521a4fe7", "patient_id": "sub-olf26", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf26/anat/sub-olf26_run-01_T1w.nii.gz"}, {"path": "sub-olf26/anat/sub-olf26_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf26/anat/sub-olf26_run-02_T1w.nii.gz", "bytes": 28313576, "checksum": "sha256:bcf3f46ef4e432d78ea4060c24e8a3ee981e46abe103007e7677146342fcf238", "patient_id": "sub-olf26", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf26/anat/sub-olf26_run-02_T1w.nii.gz"}, {"path": "sub-olf27/anat/sub-olf27_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf27/anat/sub-olf27_T2w.nii.gz", "bytes": 54959472, "checksum": "sha256:3e5819af173c97d7ce39539d5473119b5b57f6a46c5e523c2a427bf27f7c004d", "patient_id": "sub-olf27", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf27/anat/sub-olf27_T2w.nii.gz"}, {"path": "sub-olf27/anat/sub-olf27_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf27/anat/sub-olf27_run-01_T1w.nii.gz", "bytes": 27176812, "checksum": "sha256:7e288525644ec59e615b1c8d255fc54af422805745a9dea3087c5cfddf81fbb5", "patient_id": "sub-olf27", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf27/anat/sub-olf27_run-01_T1w.nii.gz"}, {"path": "sub-olf27/anat/sub-olf27_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf27/anat/sub-olf27_run-02_T1w.nii.gz", "bytes": 30153801, "checksum": "sha256:816da1cd1b45538206dde7587a60af95741df4d72bd34b641052e01e92ab686e", "patient_id": "sub-olf27", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf27/anat/sub-olf27_run-02_T1w.nii.gz"}, {"path": "sub-olf28/anat/sub-olf28_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf28/anat/sub-olf28_T2w.nii.gz", "bytes": 66100370, "checksum": "sha256:50379ca14dd5c547e61d3c44ee768f4ffa0f9e5cf054fcfc7a2bcfb729c5ec4b", "patient_id": "sub-olf28", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf28/anat/sub-olf28_T2w.nii.gz"}, {"path": "sub-olf28/anat/sub-olf28_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf28/anat/sub-olf28_run-01_T1w.nii.gz", "bytes": 29348571, "checksum": "sha256:a21d9a23292421f2dd976dd8ac1ed800b7b3cd261805303a951968c9b1a88975", "patient_id": "sub-olf28", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf28/anat/sub-olf28_run-01_T1w.nii.gz"}, {"path": "sub-olf28/anat/sub-olf28_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf28/anat/sub-olf28_run-02_T1w.nii.gz", "bytes": 31684769, "checksum": "sha256:f794ae449d13d43592d15198f5b8c23844aa98da19cca373aa6898b127f9df24", "patient_id": "sub-olf28", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf28/anat/sub-olf28_run-02_T1w.nii.gz"}, {"path": "sub-olf29/anat/sub-olf29_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf29/anat/sub-olf29_T2w.nii.gz", "bytes": 56461067, "checksum": "sha256:71bba79eaa13af44bce127cb8d7ce754ddb6bae43a8ddd284f54f6115b975483", "patient_id": "sub-olf29", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf29/anat/sub-olf29_T2w.nii.gz"}, {"path": "sub-olf29/anat/sub-olf29_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf29/anat/sub-olf29_run-01_T1w.nii.gz", "bytes": 26471615, "checksum": "sha256:d99dd87678feaeeea4e98544773b8063f3603a78a430430dc6cbf745fedfdf43", "patient_id": "sub-olf29", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf29/anat/sub-olf29_run-01_T1w.nii.gz"}, {"path": "sub-olf29/anat/sub-olf29_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf29/anat/sub-olf29_run-02_T1w.nii.gz", "bytes": 29085874, "checksum": "sha256:67626170bbda522c6944aa0acc5b3f89aebaa541c23f7ea25e791a66487f0284", "patient_id": "sub-olf29", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf29/anat/sub-olf29_run-02_T1w.nii.gz"}, {"path": "sub-olf30/anat/sub-olf30_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf30/anat/sub-olf30_T2w.nii.gz", "bytes": 53280720, "checksum": "sha256:7bbfb0b3c3d6c6044f4926eb0070b4ea61c60a09c01109558c2746c7f9815bbc", "patient_id": "sub-olf30", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf30/anat/sub-olf30_T2w.nii.gz"}, {"path": "sub-olf30/anat/sub-olf30_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf30/anat/sub-olf30_run-01_T1w.nii.gz", "bytes": 28907064, "checksum": "sha256:4a465b4ef57a83295363ddd5f40aac670fdc6f337c6dd64b8d66c5b1eb25e2c5", "patient_id": "sub-olf30", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf30/anat/sub-olf30_run-01_T1w.nii.gz"}, {"path": "sub-olf30/anat/sub-olf30_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf30/anat/sub-olf30_run-02_T1w.nii.gz", "bytes": 32120404, "checksum": "sha256:cd5dde1a0d5c260603b877f87f64d98035366ddc6c55e8be9f7e8bbe508aa7ef", "patient_id": "sub-olf30", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf30/anat/sub-olf30_run-02_T1w.nii.gz"}, {"path": "sub-olf31/anat/sub-olf31_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf31/anat/sub-olf31_T2w.nii.gz", "bytes": 59246149, "checksum": "sha256:b8edbaa13ce983b34e1cc5fb9280477a295cb7e95c53363aa86ee8cbcd9e2f27", "patient_id": "sub-olf31", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf31/anat/sub-olf31_T2w.nii.gz"}, {"path": "sub-olf31/anat/sub-olf31_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf31/anat/sub-olf31_run-01_T1w.nii.gz", "bytes": 28422327, "checksum": "sha256:9f4bb21e28de2d4f3899e4d1f1efe4536b4b1ef8ccc340a4bd69c293946e4fb4", "patient_id": "sub-olf31", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf31/anat/sub-olf31_run-01_T1w.nii.gz"}, {"path": "sub-olf31/anat/sub-olf31_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf31/anat/sub-olf31_run-02_T1w.nii.gz", "bytes": 31064301, "checksum": "sha256:d86a806fe65c7946cf1d62806622644f1023e0063d113bed961ac2553df400fb", "patient_id": "sub-olf31", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf31/anat/sub-olf31_run-02_T1w.nii.gz"}, {"path": "sub-olf32/anat/sub-olf32_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf32/anat/sub-olf32_T2w.nii.gz", "bytes": 53025240, "checksum": "sha256:e1557f216464bfd34ac88bb423917758a46d0e753f33f8d0df925604e0e4c67c", "patient_id": "sub-olf32", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf32/anat/sub-olf32_T2w.nii.gz"}, {"path": "sub-olf32/anat/sub-olf32_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf32/anat/sub-olf32_run-01_T1w.nii.gz", "bytes": 25341713, "checksum": "sha256:5b79d66047a18eefc0354902e1fe7862eb28a388bba6277b13a08d7274dd33b3", "patient_id": "sub-olf32", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf32/anat/sub-olf32_run-01_T1w.nii.gz"}, {"path": "sub-olf32/anat/sub-olf32_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf32/anat/sub-olf32_run-02_T1w.nii.gz", "bytes": 29050221, "checksum": "sha256:dcb7669837f72a47bf4812e287a65e2c1986c7ac40755c09c6d5f4d8d490fd8c", "patient_id": "sub-olf32", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf32/anat/sub-olf32_run-02_T1w.nii.gz"}, {"path": "sub-olf33/anat/sub-olf33_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf33/anat/sub-olf33_T2w.nii.gz", "bytes": 56493574, "checksum": "sha256:bf6f628b4b5f9419eea6c919c80d0681b72379a080a376713537b23ba369e564", "patient_id": "sub-olf33", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf33/anat/sub-olf33_T2w.nii.gz"}, {"path": "sub-olf33/anat/sub-olf33_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf33/anat/sub-olf33_run-01_T1w.nii.gz", "bytes": 27943105, "checksum": "sha256:16f53c6517a34c06b92be6ef705490c76f512b3d08bea201b3516b7a4c7faaaa", "patient_id": "sub-olf33", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf33/anat/sub-olf33_run-01_T1w.nii.gz"}, {"path": "sub-olf33/anat/sub-olf33_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf33/anat/sub-olf33_run-02_T1w.nii.gz", "bytes": 30955360, "checksum": "sha256:02c87f36adb0e7c9d183d79d8970d35c2bc1bd2f546b1feb73de713edc507668", "patient_id": "sub-olf33", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf33/anat/sub-olf33_run-02_T1w.nii.gz"}, {"path": "sub-olf34/anat/sub-olf34_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf34/anat/sub-olf34_T2w.nii.gz", "bytes": 54838757, "checksum": "sha256:10b4bc58e1ed973734e8498c07df17b6eca68d2241d21e245c250de661ec4232", "patient_id": "sub-olf34", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf34/anat/sub-olf34_T2w.nii.gz"}, {"path": "sub-olf34/anat/sub-olf34_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf34/anat/sub-olf34_run-01_T1w.nii.gz", "bytes": 28243868, "checksum": "sha256:8dd6756dba2a59e6462d952fa4f035a7973ec12cdb97192a3f1b0a475ce33a79", "patient_id": "sub-olf34", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf34/anat/sub-olf34_run-01_T1w.nii.gz"}, {"path": "sub-olf34/anat/sub-olf34_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf34/anat/sub-olf34_run-02_T1w.nii.gz", "bytes": 32642732, "checksum": "sha256:ee16f7f9edd911b426982b9d56e00118dd7e706567a03402d9f3364c87e789ec", "patient_id": "sub-olf34", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf34/anat/sub-olf34_run-02_T1w.nii.gz"}, {"path": "sub-olf35/anat/sub-olf35_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf35/anat/sub-olf35_T2w.nii.gz", "bytes": 59070935, "checksum": "sha256:9738376d78d92ca38a78be7656362c499a2d1c22e678da83a5e76d86fcd9e09f", "patient_id": "sub-olf35", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf35/anat/sub-olf35_T2w.nii.gz"}, {"path": "sub-olf35/anat/sub-olf35_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf35/anat/sub-olf35_run-01_T1w.nii.gz", "bytes": 29102922, "checksum": "sha256:84c3b48f9d6d63f77bdd218a845ad53f68f9982b091c397dcf485f589b327e4f", "patient_id": "sub-olf35", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf35/anat/sub-olf35_run-01_T1w.nii.gz"}, {"path": "sub-olf35/anat/sub-olf35_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf35/anat/sub-olf35_run-02_T1w.nii.gz", "bytes": 31274432, "checksum": "sha256:5cb9bf8f4cd5baaa555faf5fa4ae58f688e57d9d9fcebc61c26a839356f0da3d", "patient_id": "sub-olf35", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf35/anat/sub-olf35_run-02_T1w.nii.gz"}, {"path": "sub-olf36/anat/sub-olf36_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf36/anat/sub-olf36_T2w.nii.gz", "bytes": 66836810, "checksum": "sha256:85a253cf671db94ac629a64776762a6e26806dc6ae6153bdfe20000624953ac9", "patient_id": "sub-olf36", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf36/anat/sub-olf36_T2w.nii.gz"}, {"path": "sub-olf36/anat/sub-olf36_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf36/anat/sub-olf36_run-01_T1w.nii.gz", "bytes": 34185108, "checksum": "sha256:bb8d20f7bacee56906835e623f920f73c4986ee07acf2952081965933044d76f", "patient_id": "sub-olf36", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf36/anat/sub-olf36_run-01_T1w.nii.gz"}, {"path": "sub-olf36/anat/sub-olf36_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf36/anat/sub-olf36_run-02_T1w.nii.gz", "bytes": 38918522, "checksum": "sha256:423bf5447160574164d616d48095201911fbe16548da59262948b3a2d12ff8ef", "patient_id": "sub-olf36", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf36/anat/sub-olf36_run-02_T1w.nii.gz"}, {"path": "sub-olf37/anat/sub-olf37_T2w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf37/anat/sub-olf37_T2w.nii.gz", "bytes": 62621345, "checksum": "sha256:561232199258838846d6da7fff3cda3db85e8a44294aa14ef56032c1bb7df69d", "patient_id": "sub-olf37", "role": "image", "modality": "T2w", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf37/anat/sub-olf37_T2w.nii.gz"}, {"path": "sub-olf37/anat/sub-olf37_run-01_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf37/anat/sub-olf37_run-01_T1w.nii.gz", "bytes": 33733678, "checksum": "sha256:73fefa7b134385ae2b8f577d82a7d97699b80698b3974952637e0ff641877ad1", "patient_id": "sub-olf37", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf37/anat/sub-olf37_run-01_T1w.nii.gz"}, {"path": "sub-olf37/anat/sub-olf37_run-02_T1w.nii.gz", "url": "https://s3.amazonaws.com/openneuro.org/ds007345/sub-olf37/anat/sub-olf37_run-02_T1w.nii.gz", "bytes": 37670736, "checksum": "sha256:881e0fb11a1034c7856f23130c935e488c0a23628edc292fa4ec1551cece8adc", "patient_id": "sub-olf37", "role": "image", "modality": "T1w_contrast_unspecified", "pointer_source": "https://raw.githubusercontent.com/OpenNeuroDatasets/ds007345/fd052d49bfa51677fca817efdb68f237eeda9774/sub-olf37/anat/sub-olf37_run-02_T1w.nii.gz"}], "limitations": ["Single center, olfactory groove subtype only; not representative of all meningiomas.", "Only baseline structural images and available tumor/edema masks selected; DWI, fMRI, processed diffusion and all postoperative repeats excluded.", "T1 run numbers do not reliably establish pre/post contrast; retain contrast_unspecified until source metadata review.", "Edema masks are not supplied for every patient; missing is not a healthy label."], "expected_min_masks": 37}}}, "brain_metastasis": {"id": "proteas_23171182", "disease": "Brain Metastasis", "eligible": true, "selection": "Author-deposited, ungated latest PROTEAS release; replaces unresolved UCSF transfer", "source": "https://zenodo.org/records/23171182", "metadata_source": "https://zenodo.org/api/records/23171182", "release_checked_on": "2026-10-22", "previous_record": "17253793", "concept_record": "15499111", "latest_resolution": "https://zenodo.org/api/records/17253793/versions/latest", "license": "CC-BY-4.0", "license_url": "https://creativecommons.org/licenses/by/4.0/", "citation": "Flouri et al. A longitudinal MRI dataset of brain metastases with tumor segmentations, clinical & radiomic data. Scientific Data 12, 1828 (2025). doi:10.1038/s41597-025-06131-0; data doi:10.5281/zenodo.23171182", "independent_patients": 40, "expected_groups": 40, "expected_min_masks": 40, "image_studies": 185, "metastases": 65, "modalities": ["T1w", "T1ce", "T2w", "FLAIR"], "formats": ["NIfTI (.nii.gz) in BraTS space", "raw DICOM", "clinical XLSX", "CT and radiotherapy NIfTI ancillary data"], "download_bytes": 15131636697, "download_probe": "All 45 patient/course ZIPs and clinical XLSX returned anonymous HEAD 200 with matching sizes; Zenodo publisher MD5 supplied. No MRI payload downloaded.", "assets": [{"path": "P02.zip", "url": "https://zenodo.org/api/records/23171182/files/P02.zip/content", "bytes": 172566048, "checksum": "md5:b998b27c764fb68f09503100f4e32032", "archive": true, "extract_subdir": "P02"}, {"path": "P16.zip", "url": "https://zenodo.org/api/records/23171182/files/P16.zip/content", "bytes": 204662397, "checksum": "md5:fec0d397af63b605de917188cfa3727c", "archive": true, "extract_subdir": "P16"}, {"path": "P10.zip", "url": "https://zenodo.org/api/records/23171182/files/P10.zip/content", "bytes": 332719926, "checksum": "md5:273724f7537af88b34c4fc9cd0080d5b", "archive": true, "extract_subdir": "P10"}, {"path": "P06.zip", "url": "https://zenodo.org/api/records/23171182/files/P06.zip/content", "bytes": 243703717, "checksum": "md5:04400bc67293226ad6468fbb60ac3cd0", "archive": true, "extract_subdir": "P06"}, {"path": "P07b.zip", "url": "https://zenodo.org/api/records/23171182/files/P07b.zip/content", "bytes": 238574188, "checksum": "md5:fe479dc02f0d48dda848ddf840c67f41", "archive": true, "extract_subdir": "P07b"}, {"path": "P09.zip", "url": "https://zenodo.org/api/records/23171182/files/P09.zip/content", "bytes": 396605629, "checksum": "md5:e6ccdf4400e1e1272ceff5d3083bedd2", "archive": true, "extract_subdir": "P09"}, {"path": "P08.zip", "url": "https://zenodo.org/api/records/23171182/files/P08.zip/content", "bytes": 389292284, "checksum": "md5:e20d9283fd1299e5035c323207e7b4bd", "archive": true, "extract_subdir": "P08"}, {"path": "P21.zip", "url": "https://zenodo.org/api/records/23171182/files/P21.zip/content", "bytes": 178974119, "checksum": "md5:58e106da2200facb4fb6788cf6f7bb75", "archive": true, "extract_subdir": "P21"}, {"path": "P17a.zip", "url": "https://zenodo.org/api/records/23171182/files/P17a.zip/content", "bytes": 206292890, "checksum": "md5:9fc97a538287335a880ce48de06ac866", "archive": true, "extract_subdir": "P17a"}, {"path": "P27.zip", "url": "https://zenodo.org/api/records/23171182/files/P27.zip/content", "bytes": 247540754, "checksum": "md5:a5fc5e9271fd9c6f80c29545a7858183", "archive": true, "extract_subdir": "P27"}, {"path": "P11.zip", "url": "https://zenodo.org/api/records/23171182/files/P11.zip/content", "bytes": 383234053, "checksum": "md5:42eb6797befa0232b2cf74260a70c67c", "archive": true, "extract_subdir": "P11"}, {"path": "P35.zip", "url": "https://zenodo.org/api/records/23171182/files/P35.zip/content", "bytes": 446150618, "checksum": "md5:b91260a345a397d676fe4a15ce0f7915", "archive": true, "extract_subdir": "P35"}, {"path": "P13.zip", "url": "https://zenodo.org/api/records/23171182/files/P13.zip/content", "bytes": 251433430, "checksum": "md5:e9588bb2e37215423ddb0f120663ef10", "archive": true, "extract_subdir": "P13"}, {"path": "P30.zip", "url": "https://zenodo.org/api/records/23171182/files/P30.zip/content", "bytes": 343228247, "checksum": "md5:9aae3f01e9c885fdc0e8cb48b789aeb6", "archive": true, "extract_subdir": "P30"}, {"path": "P14.zip", "url": "https://zenodo.org/api/records/23171182/files/P14.zip/content", "bytes": 140303794, "checksum": "md5:c911532a819ba5d0404cddc429c801c6", "archive": true, "extract_subdir": "P14"}, {"path": "P18.zip", "url": "https://zenodo.org/api/records/23171182/files/P18.zip/content", "bytes": 321305701, "checksum": "md5:95a220764f467a28811b39060d90f41e", "archive": true, "extract_subdir": "P18"}, {"path": "P20b.zip", "url": "https://zenodo.org/api/records/23171182/files/P20b.zip/content", "bytes": 322029039, "checksum": "md5:90b2f0f6272e8a1a80f191085f1a0220", "archive": true, "extract_subdir": "P20b"}, {"path": "P19.zip", "url": "https://zenodo.org/api/records/23171182/files/P19.zip/content", "bytes": 345481674, "checksum": "md5:9880dcece5ee24decfde6f63b1d89672", "archive": true, "extract_subdir": "P19"}, {"path": "P29.zip", "url": "https://zenodo.org/api/records/23171182/files/P29.zip/content", "bytes": 677250446, "checksum": "md5:17633043f5a4c023876fd1c2b8c3f89a", "archive": true, "extract_subdir": "P29"}, {"path": "P20a.zip", "url": "https://zenodo.org/api/records/23171182/files/P20a.zip/content", "bytes": 332652150, "checksum": "md5:22fb5bac78072cfa504cf2ce0987d9d5", "archive": true, "extract_subdir": "P20a"}, {"path": "P22.zip", "url": "https://zenodo.org/api/records/23171182/files/P22.zip/content", "bytes": 324884036, "checksum": "md5:54473f96d57b1690fc4a29339911f069", "archive": true, "extract_subdir": "P22"}, {"path": "P23a.zip", "url": "https://zenodo.org/api/records/23171182/files/P23a.zip/content", "bytes": 253337814, "checksum": "md5:7b24f4991e4cf1991bef0a5d8cade740", "archive": true, "extract_subdir": "P23a"}, {"path": "P23b.zip", "url": "https://zenodo.org/api/records/23171182/files/P23b.zip/content", "bytes": 381315713, "checksum": "md5:4c4774eefa0a2da807ebecab5f0f0dd3", "archive": true, "extract_subdir": "P23b"}, {"path": "P26.zip", "url": "https://zenodo.org/api/records/23171182/files/P26.zip/content", "bytes": 199741910, "checksum": "md5:3339f3fd536ab598a49da09ba41e4f17", "archive": true, "extract_subdir": "P26"}, {"path": "P24.zip", "url": "https://zenodo.org/api/records/23171182/files/P24.zip/content", "bytes": 173199165, "checksum": "md5:63ce32114335834981418087ed1e7340", "archive": true, "extract_subdir": "P24"}, {"path": "P25.zip", "url": "https://zenodo.org/api/records/23171182/files/P25.zip/content", "bytes": 306357163, "checksum": "md5:80b38de204b4ccc5d16e4b3273efc422", "archive": true, "extract_subdir": "P25"}, {"path": "P40.zip", "url": "https://zenodo.org/api/records/23171182/files/P40.zip/content", "bytes": 152954595, "checksum": "md5:9bdf542ce32af9b4c7ef979d10bd24e3", "archive": true, "extract_subdir": "P40"}, {"path": "P31.zip", "url": "https://zenodo.org/api/records/23171182/files/P31.zip/content", "bytes": 399026528, "checksum": "md5:19b8777f4c8ecf078cf5f9a3a2d071ee", "archive": true, "extract_subdir": "P31"}, {"path": "P34.zip", "url": "https://zenodo.org/api/records/23171182/files/P34.zip/content", "bytes": 616588199, "checksum": "md5:2f9dc4cfb41523a5345d2e93fa111e7b", "archive": true, "extract_subdir": "P34"}, {"path": "P36.zip", "url": "https://zenodo.org/api/records/23171182/files/P36.zip/content", "bytes": 583818331, "checksum": "md5:48cb87c6688ac16567e6c06cfde5f580", "archive": true, "extract_subdir": "P36"}, {"path": "P33.zip", "url": "https://zenodo.org/api/records/23171182/files/P33.zip/content", "bytes": 585573230, "checksum": "md5:90f5db1c8fce8f4e0ca6f6c803c84929", "archive": true, "extract_subdir": "P33"}, {"path": "P37.zip", "url": "https://zenodo.org/api/records/23171182/files/P37.zip/content", "bytes": 403042742, "checksum": "md5:0cd364d0a284cb3fefe69c0746fa0998", "archive": true, "extract_subdir": "P37"}, {"path": "P39.zip", "url": "https://zenodo.org/api/records/23171182/files/P39.zip/content", "bytes": 311416437, "checksum": "md5:c10897f37aa4975c820eb44836b679ef", "archive": true, "extract_subdir": "P39"}, {"path": "P38.zip", "url": "https://zenodo.org/api/records/23171182/files/P38.zip/content", "bytes": 417641609, "checksum": "md5:f2698ad65d1b968a1654ac0742be1635", "archive": true, "extract_subdir": "P38"}, {"path": "P32.zip", "url": "https://zenodo.org/api/records/23171182/files/P32.zip/content", "bytes": 780265759, "checksum": "md5:33eda2c7be4a3f1d6d4df35d39df9587", "archive": true, "extract_subdir": "P32"}, {"path": "P12.zip", "url": "https://zenodo.org/api/records/23171182/files/P12.zip/content", "bytes": 186138149, "checksum": "md5:9a5b33b4526161198a2efba1c4201490", "archive": true, "extract_subdir": "P12"}, {"path": "P15.zip", "url": "https://zenodo.org/api/records/23171182/files/P15.zip/content", "bytes": 326204070, "checksum": "md5:7f6354ac961678ae00ccc5fb7c3d9b9f", "archive": true, "extract_subdir": "P15"}, {"path": "P04a.zip", "url": "https://zenodo.org/api/records/23171182/files/P04a.zip/content", "bytes": 207177098, "checksum": "md5:b6745ad76dbcf580d1bb4b7a35fed2f1", "archive": true, "extract_subdir": "P04a"}, {"path": "P07a.zip", "url": "https://zenodo.org/api/records/23171182/files/P07a.zip/content", "bytes": 237126654, "checksum": "md5:1ed37207c1469120f6218d7013afc6dd", "archive": true, "extract_subdir": "P07a"}, {"path": "P01.zip", "url": "https://zenodo.org/api/records/23171182/files/P01.zip/content", "bytes": 286009271, "checksum": "md5:6f8c6ba98a275d4a7864f4d3cc5f26a2", "archive": true, "extract_subdir": "P01"}, {"path": "P04b.zip", "url": "https://zenodo.org/api/records/23171182/files/P04b.zip/content", "bytes": 303026162, "checksum": "md5:e8c0d019c913c8e796299af5a568209a", "archive": true, "extract_subdir": "P04b"}, {"path": "P03.zip", "url": "https://zenodo.org/api/records/23171182/files/P03.zip/content", "bytes": 216958958, "checksum": "md5:b3b8672949372c70abb5ebc0409da519", "archive": true, "extract_subdir": "P03"}, {"path": "P05.zip", "url": "https://zenodo.org/api/records/23171182/files/P05.zip/content", "bytes": 242549302, "checksum": "md5:0dc4732abe7760898224e00fcca2eb97", "archive": true, "extract_subdir": "P05"}, {"path": "P28.zip", "url": "https://zenodo.org/api/records/23171182/files/P28.zip/content", "bytes": 792393311, "checksum": "md5:0ffb2efd46a3861831439f964feb21c6", "archive": true, "extract_subdir": "P28"}, {"path": "PROTEAS-Clinical_and_demographic_data.xlsx", "url": "https://zenodo.org/api/records/23171182/files/PROTEAS-Clinical_and_demographic_data.xlsx/content", "bytes": 29672, "checksum": "md5:848bfded9d4746787001c7475faaf644", "archive": false}, {"path": "P17b.zip", "url": "https://zenodo.org/api/records/23171182/files/P17b.zip/content", "bytes": 270859715, "checksum": "md5:64811d23ac95f8cfa84eee4b5b904db2", "archive": true, "extract_subdir": "P17b"}], "limitations": ["Single-center treatment-enriched cohort; pre/post-treatment scans are not independent patients.", "45 course archives represent 40 patients. P04/P07/P17/P20/P23 a/b courses share one patient split group; P31/P33 lesion rows in clinical metadata do not create new patients.", "Six ZIP MD5s and the clinical spreadsheet changed since 17253793. Never merge old and new release files.", "Lesions under 3 mm were excluded by the study. Masks include necrosis=1, enhancing tumor=2, edema=3; tissue labels 10/30/40/50 are not tumor labels.", "ZIPs also contain CT, radiotherapy and raw DICOM. MRI manifests use only documented BraTS MRI sequences; other NIfTI data are labeled ancillary. No conversion or model preprocessing runs.", "Radiomics XLSX is not needed for acquisition and is not selected."]}, "alzheimers": {"id": "oasis2_legacy", "disease": "Alzheimer\'s Disease", "eligible": true, "selection": "Official legacy OASIS-2 direct archives, with explicit clinical-group review and weaker integrity disclosure", "source": "https://www.nitrc.org/projects/oasis/", "authorized_access_evidence": "https://github.com/NrgXnat/oasis-scripts#oasis-scripts-overview", "license": "OASIS Data Use Terms (custom; not CC0 or CC BY)", "license_url": "https://www.nitrc.org/projects/oasis/", "license_summary": "Scientific use; no reidentification/contact or facial reconstruction; source-specific acknowledgement/citation; provide Knight ADRC usage information on request. PET-specific Avid review terms do not apply to this T1-only selection.", "citation": "Marcus et al. Open Access Series of Imaging Studies (OASIS): Longitudinal MRI Data in Nondemented and Demented Older Adults. Journal of Cognitive Neuroscience 22(12):2677-2684 (2010). doi:10.1162/jocn.2009.21407", "paper_source": "https://pmc.ncbi.nlm.nih.gov/articles/PMC2895005/", "independent_patients": 150, "expected_groups": 150, "clinical_groups": {"AD_at_baseline": 64, "stable_nondemented_controls": 72, "converters_to_AD_exclude_from_binary_training": 14}, "expected_diagnosis_groups": {"AD_clinical_baseline": 64, "control_stable": 72, "converter_excluded": 14}, "sessions": 373, "modalities": ["T1w MP-RAGE, 1.5 T"], "formats": ["NIfTI-1; paired IMG/HDR supported; exact raw packaging inspected in Colab"], "download_bytes": 19323750662, "download_probe": "Both official raw archives returned anonymous HEAD 200 and Range support. Authorized maintainer README explicitly says OASIS1/2 download directly; later OASIS releases need signed DUA/project access.", "assets": [{"path": "OAS2_RAW_PART1.tar.gz", "url": "https://download.nrg.wustl.edu/data/OAS2_RAW_PART1.tar.gz", "bytes": 10874461059, "checksum": null, "integrity_policy": "https_size_gzip_crc32_local_sha256", "archive": true}, {"path": "OAS2_RAW_PART2.tar.gz", "url": "https://download.nrg.wustl.edu/data/OAS2_RAW_PART2.tar.gz", "bytes": 8449289603, "checksum": null, "integrity_policy": "https_size_gzip_crc32_local_sha256", "archive": true}], "patient_map_required": true, "limitations": ["No publisher cryptographic checksum is available from the official directory or HEAD metadata. HTTPS, exact source size and full gzip CRC32 are verified; local SHA-256 is saved and compared on repeat use. This is weaker than publisher-hash verification and is explicitly reported before download.", "The current central website returns 503; access/terms are corroborated by the publisher-maintained NITRC page, official NrgXnat scripts and official Washington University download server, not third-party MRI copies.", "Three or four T1 acquisitions per session and repeated visits are not independent patients. Namespaced OAS2 identifiers group every visit/channel together.", "Clinical AD diagnosis follows this cohort\'s published assessment, not an arbitrary dementia or MCI folder label. Do not equate generic CDR 0.5 with AD.", "A reviewed per-file clinical-group map is required. The 14 converters must be marked converter_excluded, not mixed into stable control/AD groups. Missing diagnosis remains unknown. The notebook does not fabricate clinical metadata.", "Without a reviewed map the download can finish and patient identities can be inventoried, but diagnostic labels remain pending. No binary AD/control training readiness is claimed.", "Single-scanner historical research sample; clinical AD is not equivalent to universal biomarker confirmation. No focal lesion masks. Existing IXI is not used or changed."], "required_acknowledgement": "Data provided [in part] by OASIS. OASIS-2: Longitudinal: Principal Investigators: D. Marcus, R. Buckner, J. Csernansky, J. Morris; P50 AG05681, P01 AG03991, P01 AG026276, R01 AG021910, P20 MH071616, U24 RR021382."}, "post_stroke": {"id": "isles2022", "disease": "Post-stroke Lesions", "eligible": true, "selection": "Verified public multi-center annotated stroke alternative; acute lesions, not chronic T1-only ATLAS", "source": "https://zenodo.org/records/7153326", "metadata_source": "https://zenodo.org/api/records/7153326", "license": "CC-BY-4.0", "license_url": "https://creativecommons.org/licenses/by/4.0/", "citation": "Hernandez Petzsche et al. ISLES 2022: A multi-center magnetic resonance imaging stroke lesion segmentation dataset. Scientific Data 9, 762 (2022). doi:10.1038/s41597-022-01875-5; dataset doi:10.5281/zenodo.7153326", "independent_patients": 250, "expected_groups": 250, "download_bytes": 1692717470, "download_probe": "HEAD 200 with matching 1692717470 byte size. Official open-access Zenodo record and publisher MD5 verified; no payload fetched in Cloud.", "assets": [{"path": "ISLES-2022.zip", "url": "https://zenodo.org/api/records/7153326/files/ISLES-2022.zip/content", "bytes": 1692717470, "checksum": "md5:302ee280373cdd5c190ab763d72a7a50", "archive": true}], "limitations": ["250 public training cases, not the 400-case full challenge cohort.", "Acute/subacute ischemic lesions; not a substitute for chronic post-stroke T1 structural lesions.", "FLAIR, DWI and ADC remain in native space; do not assume cross-sequence registration.", "No normal/control subjects; IXI differs in modality, age distribution and scanner protocol."], "expected_min_masks": 250}}')


In [ ]:
"""Colab-only acquisition helpers. Importing this module never downloads data."""
from __future__ import annotations

import csv
import http.client
import hashlib
import gzip
import json
import os
import re
import shutil
import stat
import tarfile
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath


def digest(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    content = json.dumps(value, indent=2) + "\n"
    if path.is_symlink():
        raise ValueError(f"Refusing metadata symlink: {path}")
    if path.exists():
        if path.read_text() == content:
            return
        # Retain prior provenance and failure reports on reruns.
        backup = path.with_name(path.name + f".previous.{time.time_ns()}")
        with path.open("rb") as source, backup.open("xb") as output:
            shutil.copyfileobj(source, output)
    with tempfile.NamedTemporaryFile(mode="w", dir=path.parent, prefix=path.name + ".", suffix=".tmp", delete=False) as stream:
        stream.write(content)
        tmp = Path(stream.name)
    tmp.replace(path)


def checked_path(root, relative):
    if Path(root).is_symlink():
        raise ValueError(f"Refusing destination directory symlink: {root}")
    root = Path(root).resolve()
    p = PurePosixPath(relative)
    if not p.parts or p.is_absolute() or ".." in p.parts or "\\" in relative or ":" in relative:
        raise ValueError(f"Unsafe archive/path member: {relative}")
    target = root.joinpath(*p.parts)
    if not target.resolve().is_relative_to(root):
        raise ValueError(f"Path escapes destination: {relative}")
    return target


def is_drive_path(root):
    return Path(root).resolve().is_relative_to(Path("/content/drive").resolve())


def storage_reservation(download_bytes, unpacked_bytes):
    """Retained archives + extracted outputs coexist; reserve 10% or 5 GiB extra."""
    if min(download_bytes, unpacked_bytes) < 0:
        raise ValueError("Storage budgets must be nonnegative")
    footprint = download_bytes + unpacked_bytes
    return footprint + max(5 * 2**30, (footprint + 9) // 10)


def require_space(root, needed, reserve=2 * 1024**3, *, drive_available_bytes=None):
    if needed < 0 or reserve < 0:
        raise ValueError("Storage requirements must be nonnegative")
    if is_drive_path(root):
        # DriveFS statvfs may expose a local/cache capacity, not account quota.
        if drive_available_bytes is None or drive_available_bytes <= 0:
            raise ValueError("A current user-confirmed Google Drive free quota is required")
        free = drive_available_bytes
        source = "user-confirmed Google Drive quota"
    else:
        free = shutil.disk_usage(root).free
        source = "local filesystem"
    if free < needed + reserve:
        raise OSError(f"Insufficient {source} space: need {needed + reserve:,} bytes; free {free:,}")


def check_storage_plan(root, download_bytes, unpacked_bytes, *, drive_available_bytes, local_root=None, local_staging_bytes=0):
    """Explicit separate accounting. Streaming acquisition stages no MRI on local disk.

    Any future local staging must declare its simultaneous peak bytes here.
    The local reserve also leaves room for runtime/cache overhead; DriveFS may still
    fail writes for provider/cache limits, which must propagate as acquisition failures.
    """
    reservation = storage_reservation(download_bytes, unpacked_bytes)
    require_space(root, reservation, reserve=0, drive_available_bytes=drive_available_bytes)
    if local_root is not None:
        if is_drive_path(local_root):
            raise ValueError("Local storage check must target real Colab local disk")
        require_space(local_root, local_staging_bytes)
    return {"reserved_bytes": reservation, "download_bytes": download_bytes,
            "unpacked_budget_bytes": unpacked_bytes, "headroom_bytes": reservation - download_bytes - unpacked_bytes,
            "drive_available_bytes": drive_available_bytes, "local_staging_bytes": local_staging_bytes,
            "local_runtime_reserve_bytes": 2 * 2**30 if local_root is not None else 0}


def assert_colab_drive(root):
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("Dataset acquisition is allowed only in Google Colab.") from exc
    mount = Path("/content/drive/MyDrive")
    root = Path(root).resolve()
    if not mount.is_dir() or not root.is_relative_to(mount.resolve()):
        raise RuntimeError("Mount Drive and choose a destination inside /content/drive/MyDrive.")
    # Protect the entire existing IXI tree, not just its Healthy/T2 child.
    protected = Path("/content/drive/MyDrive/BrainMRI/datasets/IXI").resolve()
    if root == protected or root.is_relative_to(protected) or protected.is_relative_to(root):
        raise ValueError("Destination overlaps protected IXI storage.")
    return root


def _request(url, **kwargs):
    if not url.startswith("https://"):
        raise ValueError("Only verified HTTPS sources are allowed.")
    connect_timeout = kwargs.pop("connect_timeout", 30)
    read_timeout = kwargs.pop("read_timeout", 60)
    req = urllib.request.Request(url, headers={"User-Agent": "NeuroScope-Acquisition/1.0", "Accept-Encoding": "identity", **kwargs.pop("headers", {})}, **kwargs)
    response = urllib.request.urlopen(req, timeout=connect_timeout)
    try:
        response.fp.raw._sock.settimeout(read_timeout)
    except AttributeError:
        response.close()
        raise RuntimeError("Cannot configure HTTP socket read timeout")
    if not response.url.startswith("https://"):
        response.close()
        raise ValueError("Refusing a redirect that downgrades HTTPS.")
    return response


def probe_assets(spec):
    """HEAD only. A failed probe never launches a speculative payload transfer."""
    if not spec.get("eligible"):
        raise RuntimeError(spec["reason"])
    results = []
    for asset in spec["assets"]:
        with _request(asset["url"], method="HEAD") as response:
            if response.status != 200:
                raise RuntimeError(f"HEAD returned {response.status}")
            content_type = response.headers.get("Content-Type", "").lower()
            if any(t in content_type for t in ("text/html", "application/json")):
                raise RuntimeError("HEAD returned a login/error page")
            size = response.headers.get("Content-Length")
            if not size:
                raise RuntimeError("Source did not provide Content-Length; cannot preflight storage")
            size = int(size)
            if asset.get("bytes") is not None and asset["bytes"] != size:
                raise ValueError(f"Source size changed for {asset['path']}")
            asset["bytes"] = size
            results.append({"path": asset["path"], "bytes": size, "accept_ranges": response.headers.get("Accept-Ranges"), "final_url": response.url})
    return results


def record_oasis_integrity(part, ledger_path, asset):
    if not asset.get("bytes") or Path(part).stat().st_size != asset["bytes"]:
        raise ValueError("OASIS archive must match the official HEAD byte size")
    # Exhaust every gzip member to verify CRC32 and uncompressed-size trailers.
    with gzip.open(part, "rb") as stream:
        for _ in iter(lambda: stream.read(4 * 1024**2), b""):
            pass
    atomic_json(ledger_path, {"source_url": asset["url"], "bytes": asset["bytes"], "sha256": digest(part),
                             "gzip_crc32": "passed", "publisher_cryptographic_checksum": None,
                             "limitation": "Local SHA-256 is a repeat-use integrity baseline, not a publisher hash comparison"})


def _download_attempt(asset, root, *, max_bytes, drive_available_bytes=None,
                      connect_timeout=30, read_timeout=60, chunk_bytes=1024**2,
                      progress_interval=10, sync_interval=5, progress=None):
    """Resume only with a server validator and matching Content-Range; never trust ETag as MD5."""
    target = checked_path(root, asset["path"])
    checksum = asset.get("checksum")
    # This legacy publisher supplies no cryptographic digest. Keep this weaker,
    # explicit policy restricted to its two authenticated HTTPS archive URLs.
    legacy_oasis = asset.get("integrity_policy") == "https_size_gzip_crc32_local_sha256"
    if legacy_oasis and asset["url"] not in {
        "https://download.nrg.wustl.edu/data/OAS2_RAW_PART1.tar.gz",
        "https://download.nrg.wustl.edu/data/OAS2_RAW_PART2.tar.gz",
    }:
        raise ValueError("Legacy integrity policy is restricted to the two official OASIS-2 archives")
    if checksum:
        algorithm, expected = checksum.split(":", 1)
    elif legacy_oasis:
        algorithm, expected = "sha256", None
    else:
        raise ValueError("Missing publisher checksum; no explicit authorized alternative integrity policy")
    expected_size = asset.get("bytes")
    length = {"md5": 32, "sha256": 64}.get(algorithm)
    if length is None or (expected is not None and not re.fullmatch(r"[0-9a-f]{" + str(length) + r"}", expected)):
        raise ValueError("Missing/invalid publisher checksum")
    ledger_path = target.with_name(target.name + ".integrity.json")
    if target.exists():
        previous = json.loads(ledger_path.read_text()) if legacy_oasis and ledger_path.exists() else {}
        reference = expected or (previous.get("sha256") if previous.get("source_url") == asset["url"] else None)
        if reference and (expected_size is None or target.stat().st_size == expected_size) and digest(target, algorithm) == reference:
            return {"path": str(target), "sha256": digest(target), "status": "verified_existing"}
        raise FileExistsError(f"Existing file differs from publisher checksum; preserve and inspect: {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    part = target.with_name(target.name + ".part")
    state_path = target.with_name(target.name + ".resume.json")
    if part.is_symlink() or state_path.is_symlink():
        raise ValueError("Refusing partial/resume metadata symlink")
    if expected is not None and part.exists() and (expected_size is None or part.stat().st_size == expected_size) and digest(part, algorithm) == expected:
        part.replace(target)
        state_path.unlink(missing_ok=True)
        return {"path": str(target), "sha256": digest(target), "status": "verified_completed_partial"}
    try:
        state = json.loads(state_path.read_text()) if state_path.exists() else {}
    except (ValueError, TypeError):
        state = {}  # An actual partial without valid metadata remains protected below.
    if not isinstance(state, dict):
        state = {}
    if state and not part.exists():
        print(f"{asset['path']}: stale resume metadata without partial bytes; starting a fresh transfer", flush=True)
    if state.get("total") not in (None, expected_size):
        state = {}
    headers = {}
    offset = part.stat().st_size if part.exists() else 0
    if legacy_oasis and offset == expected_size and state.get("url") == asset["url"]:
        record_oasis_integrity(part, ledger_path, asset)
        part.replace(target)
        state_path.unlink(missing_ok=True)
        return {"path": str(target), "sha256": digest(target), "publisher_checksum": None,
                "integrity_policy": asset["integrity_policy"], "status": "verified_completed_partial"}
    validator = state.get("validator") if state.get("url") == asset["url"] else None
    if not isinstance(validator, str) or not validator or validator.startswith("W/"):
        validator = None
    if offset and validator:
        headers = {"Range": f"bytes={offset}-", "If-Range": validator}
    elif offset:
        raise RuntimeError(f"Partial file has no trusted resume validator; preserve it and inspect: {part}")
    with _request(asset["url"], headers=headers, connect_timeout=connect_timeout, read_timeout=read_timeout) as response:
        if response.status not in (200, 206):
            raise RuntimeError(f"Unexpected HTTP status {response.status}")
        if any(x in response.headers.get("Content-Type", "").lower() for x in ["text/html", "application/json"]):
            raise RuntimeError("Source returned a web page/API error instead of the MRI asset")
        if offset and response.status == 206:
            content_range = response.headers.get("Content-Range", "")
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
            if not match or int(match[1]) != offset:
                raise RuntimeError("Server returned an invalid resume Content-Range")
            total = int(match[3])
            if int(match[2]) != total - 1 or int(match[2]) < offset:
                raise RuntimeError("Server returned an invalid resume Content-Range end")
            response_validator = response.headers.get("ETag") if not validator.startswith('W/') and validator.startswith('"') else response.headers.get("Last-Modified")
            if response_validator and response_validator != validator:
                raise RuntimeError("Resume validator changed; partial preserved")
            if response.headers.get("Content-Length") and int(response.headers["Content-Length"]) != total - offset:
                raise RuntimeError("Resume response length does not match Content-Range")
            mode = "ab"
        else:
            if response.status == 206:
                raise RuntimeError("Unexpected partial response for a fresh download")
            if offset:
                raise RuntimeError("Server ignored Range or changed validator; reliable resume unavailable. Partial preserved; explicit restart review required")
            # A fresh request may start only when there are no existing partial bytes.
            offset = 0
            mode = "wb"
            total = int(response.headers["Content-Length"]) if response.headers.get("Content-Length") else None
        if expected_size is not None and total is not None and total != expected_size:
            raise ValueError(f"Publisher size differs: expected {expected_size}, server {total}")
        if total is not None and total > max_bytes:
            raise OSError("Asset exceeds the configured download budget")
        limit = expected_size or total or max_bytes
        if limit > max_bytes:
            raise OSError("Asset exceeds the configured download budget")
        require_space(root, max(0, limit - offset), drive_available_bytes=drive_available_bytes)
        etag = response.headers.get("ETag", "")
        validator = etag if etag and not etag.startswith("W/") else response.headers.get("Last-Modified")
        atomic_json(state_path, {"url": asset["url"], "validator": validator, "total": total})
        count = offset
        if progress:
            progress(count, "streaming", resumable=bool(validator))
        if not validator:
            print(f"{asset['path']}: server supplied no strong ETag/Last-Modified; reliable interrupted resume is unavailable", flush=True)
        last_sync = last_progress = time.monotonic()
        first = True
        # Unbuffered writes reach DriveFS immediately. fsync requests persistence;
        # remote durability still depends on DriveFS and the connected runtime.
        with part.open(mode, buffering=0) as stream:
            try:
                reader = getattr(response, "read1", response.read)
                while True:
                    chunk = reader(chunk_bytes)
                    if not chunk:
                        break
                    if count + len(chunk) > limit:
                        raise OSError("Download exceeded expected size/budget")
                    view = memoryview(chunk)
                    while view:
                        written = stream.write(view)
                        if not written:
                            raise OSError("Partial file write made no progress")
                        view = view[written:]
                        count += written
                    now = time.monotonic()
                    if first or now - last_sync >= sync_interval:
                        os.fsync(stream.fileno())
                        last_sync = now
                        first = False
                    if progress and now - last_progress >= progress_interval:
                        progress(count, "streaming", resumable=bool(validator))
                        last_progress = now
            finally:
                os.fsync(stream.fileno())
                if progress:
                    progress(count, "stream_closed", resumable=bool(validator))
        for size in (total, expected_size):
            if size is not None and count != size:
                raise ValueError(f"Incomplete download: {count} != {size}")
    if progress:
        progress(count, "verifying_checksum")
    if expected is not None and digest(part, algorithm) != expected:
        raise ValueError(f"Publisher {algorithm} mismatch; file kept as .part for investigation")
    if legacy_oasis:
        record_oasis_integrity(part, ledger_path, asset)
    part.replace(target)
    state_path.unlink(missing_ok=True)
    return {"path": str(target), "bytes": count, "publisher_checksum": checksum, "sha256": digest(target),
            "integrity_policy": asset.get("integrity_policy", "publisher_checksum"), "status": "verified"}


def download_verified(asset, root, *, max_bytes, drive_available_bytes=None,
                      connect_timeout=30, read_timeout=60, chunk_bytes=1024**2,
                      progress_interval=10, sync_interval=5, max_retries=2):
    """Bounded-memory Drive streaming; retry only transient transport/server failures.

    No unsafe restart: partials without validators and ignored Range fail closed.
    Status is informational only, never a completion/checksum authority.
    """
    if min(connect_timeout, read_timeout, progress_interval, sync_interval) <= 0 or not 1 <= chunk_bytes <= 4 * 1024**2 or not 0 <= max_retries <= 5:
        raise ValueError("Invalid transfer options (chunk <=4 MiB, retries 0..5, positive timeouts/intervals)")
    target = checked_path(root, asset["path"])
    target.parent.mkdir(parents=True, exist_ok=True)
    status_path = target.with_name(target.name + ".download_status.json")
    part = target.with_name(target.name + ".part")
    started = time.monotonic()
    initial_bytes = part.stat().st_size if part.exists() else 0
    attempt = 0
    def progress(count, status, **extra):
        report = {"status": status, "bytes_saved": count, "bytes_transferred_this_run": max(0, count - initial_bytes),
                  "expected_total": asset.get("bytes"), "elapsed_seconds": round(time.monotonic() - started, 2),
                  "attempt": attempt + 1, "url": asset["url"], **extra}
        # Replace our own small status snapshot, without accumulating progress backups.
        temporary = status_path.with_name(status_path.name + ".updating")
        if status_path.is_symlink() or temporary.is_symlink():
            raise ValueError("Unsafe download status path")
        with temporary.open("w") as stream:
            json.dump(report, stream)
            stream.flush()
            os.fsync(stream.fileno())
        temporary.replace(status_path)
        print(f"{asset['path']}: {status}; {count:,}/{asset.get('bytes')} bytes; {report['elapsed_seconds']}s; attempt {attempt + 1}", flush=True)
    for attempt in range(max_retries + 1):
        progress(part.stat().st_size if part.exists() else 0, "connecting")
        try:
            result = _download_attempt(asset, root, max_bytes=max_bytes, drive_available_bytes=drive_available_bytes,
                connect_timeout=connect_timeout, read_timeout=read_timeout, chunk_bytes=chunk_bytes,
                progress_interval=progress_interval, sync_interval=sync_interval, progress=progress)
            progress(target.stat().st_size, result["status"])
            return result
        except Exception as exc:
            transient = isinstance(exc, (TimeoutError, ConnectionError, http.client.IncompleteRead, urllib.error.URLError))
            if isinstance(exc, urllib.error.HTTPError):
                transient = exc.code in {408, 429, 500, 502, 503, 504}
            if isinstance(exc, ValueError) and str(exc).startswith("Incomplete download:"):
                transient = True
            progress(part.stat().st_size if part.exists() else 0, "interrupted" if transient else "failed",
                     error=f"{type(exc).__name__}: {exc}")
            if part.exists() and part.stat().st_size:
                resume_path = target.with_name(target.name + ".resume.json")
                try:
                    resume_state = json.loads(resume_path.read_text())
                except (OSError, ValueError):
                    resume_state = {}
                if not resume_state.get("validator"):
                    progress(part.stat().st_size, "resume_unavailable", error=f"{type(exc).__name__}: {exc}",
                             action="Partial preserved; server supplied no reliable validator. Review before explicit fresh restart")
                    raise
            if not transient or attempt == max_retries:
                raise
            time.sleep(min(2**attempt, 4))
    raise AssertionError("Unreachable")


def safe_extract(archive, destination, *, max_unpacked_bytes, drive_available_bytes=None):
    """Preflight all members; reject links, devices, traversal, duplicates and archive bombs."""
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        container = zipfile.ZipFile(archive)
        members = container.infolist()
        records = []
        for m in members:
            mode = m.external_attr >> 16
            if (stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR)) or m.flag_bits & 1:
                container.close()
                raise ValueError("ZIP contains a link, special file or encrypted entry")
            records.append((m.filename, m.file_size, m.is_dir(), m))
        opener = container.open
    else:
        container = tarfile.open(archive, "r:*")
        records = []
        for m in container.getmembers():
            if not (m.isfile() or m.isdir()):
                container.close()
                raise ValueError("TAR contains a link or special file")
            records.append((m.name, m.size, m.isdir(), m))
        opener = container.extractfile
    try:
        total = 0
        seen = set()
        for name, size, is_dir, _ in records:
            if name.rstrip("/") in (".", "") and is_dir:
                continue
            target = checked_path(destination, name)
            key = str(target.resolve()).casefold()
            if key in seen:
                raise ValueError(f"Duplicate archive member: {name}")
            seen.add(key)
            total += size
            if total > max_unpacked_bytes:
                raise OSError("Archive exceeds extraction budget")
        require_space(destination, total, drive_available_bytes=drive_available_bytes)
        for name, size, is_dir, member in records:
            if name.rstrip("/") in (".", "") and is_dir:
                continue
            target = checked_path(destination, name)
            if is_dir:
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                # Rerun/resume extraction without blindly overwriting existing data.
                with opener(member) as source:
                    h = hashlib.sha256()
                    for chunk in iter(lambda: source.read(4 * 1024**2), b""):
                        h.update(chunk)
                if target.stat().st_size == size and digest(target) == h.hexdigest():
                    continue
                raise FileExistsError(f"Extraction would replace different existing data: {target}")
            partial = target.with_name(target.name + ".extracting")
            if partial.exists():
                raise FileExistsError(f"Previous interrupted extraction requires inspection: {partial}")
            with opener(member) as source, partial.open("xb") as output:
                shutil.copyfileobj(source, output, length=4 * 1024**2)
            if partial.stat().st_size != size:
                raise ValueError(f"Extracted size mismatch: {name}")
            partial.replace(target)
        return {"members": len(records), "unpacked_bytes": total}
    finally:
        container.close()


def load_patient_map(path):
    """Explicit reviewed file-to-patient mapping; never infer diagnosis from a mask."""
    required = {"relative_path", "patient_id", "diagnosis", "modality", "role", "visit_id"}
    allowed_diagnoses = {"meningioma", "brain_lesion_negative_lung_cancer", "AD_clinical_baseline", "control_stable", "converter_excluded"}
    rows = {}
    with Path(path).open(newline="") as stream:
        reader = csv.DictReader(stream)
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"Patient map must contain {sorted(required)}")
        for row in reader:
            relative = row["relative_path"]
            if relative in rows:
                raise ValueError(f"Duplicate patient-map path: {relative}")
            if any(not row[k].strip() for k in required):
                raise ValueError(f"Incomplete reviewed identity/label mapping: {relative}")
            if row["diagnosis"] not in allowed_diagnoses or row["role"] not in {"image", "mask"}:
                raise ValueError(f"Unrecognized source diagnosis/role: {relative}")
            rows[relative] = row
    return rows


def source_identity(relative, dataset_id, patient_map=None):
    """Interpret only publisher-documented conventions; unknown identity stays unknown."""
    path = PurePosixPath(relative)
    if dataset_id == "oasis2_legacy":
        match = re.search(r"OAS2_\d{4}", relative, re.I)
        visit = re.search(r"MR\d+", relative, re.I)
        if not match or not visit:
            raise ValueError("Cannot establish OASIS-2 patient and visit from documented identifiers")
        mapping = (patient_map or {}).get(relative)
        if mapping and (mapping["patient_id"] != match[0].upper() or mapping["diagnosis"] not in {"AD_clinical_baseline", "control_stable", "converter_excluded"}):
            raise ValueError("OASIS patient map must preserve source identity and use explicit clinical cohort groups")
        if mapping and (mapping["role"] != "image" or mapping["modality"] != "T1w"):
            raise ValueError("OASIS-2 source contains T1w images, not tumor masks or other sequences")
        return {"patient": match[0].upper(), "role": "image", "modalities": ["T1w"],
                "visit_id": visit[0].upper(), "diagnosis": mapping["diagnosis"] if mapping else None,
                "diagnosis_source": "OASIS-2 clinical cohort, doi:10.1162/jocn.2009.21407; reviewed clinical mapping required; never infer AD from missing data or generic CDR/MCI"}
    if dataset_id == "proteas_23171182":
        match = re.search(r"(?:^|/)(P\d{2}[ab]?)(?:/|_)", relative)
        if not match or not 1 <= int(match[1][1:3]) <= 40:
            raise ValueError("Cannot establish PROTEAS patient from the published P01-P40 layout")
        source_case = match[1]
        patient = source_case[:3]  # a/b are separate treatment courses, not new patients.
        visit_match = re.search(r"(?:^|[/_])(baseline|fu\d+)(?:[/_.]|$)", relative, re.I)
        visit = visit_match[1].lower() if visit_match else "unspecified"
        name = path.name.lower()
        if name in {"t1.nii.gz", "t2.nii.gz", "t1c.nii.gz", "fla.nii.gz"} and "brats" in {p.lower() for p in path.parts}:
            role = "image"
            modality = {"t1.nii.gz": "T1w", "t2.nii.gz": "T2w", "t1c.nii.gz": "T1ce", "fla.nii.gz": "FLAIR"}[name]
        elif "_tumor_mask_" in name:
            role, modality = "mask", "tumor_edema_necrosis"
        elif "_brain_mask" in name:
            role, modality = "ancillary", "brain_tissue_labels"
        elif name.endswith("_ct.nii.gz"):
            role, modality = "ancillary", "CT"
        elif name.endswith("_rtp.nii.gz"):
            role, modality = "ancillary", "radiotherapy_plan"
        else:
            raise ValueError(f"Undocumented PROTEAS NIfTI file; review rather than assume MRI: {relative}")
        return {"patient": patient, "source_case": source_case, "visit_id": visit,
                "role": role, "modalities": [modality], "diagnosis": "brain_metastasis",
                "diagnosis_source": "Flouri et al. confirmed-BM cohort, doi:10.1038/s41597-025-06131-0"}
    if dataset_id == "hwang_meningioma_5945963":
        mapping = (patient_map or {}).get(relative)
        if not mapping:
            return {"patient": None, "role": "unresolved", "modalities": ["unresolved"],
                    "diagnosis": None, "visit_id": None,
                    "diagnosis_source": "Explicit source identity/diagnosis map required; absence of a mask is not a control label"}
        if mapping["diagnosis"] not in {"meningioma", "brain_lesion_negative_lung_cancer"}:
            raise ValueError("Hwang cohort requires meningioma or its source-specific lesion-negative control label")
        return {"patient": mapping["patient_id"], "role": mapping["role"], "modalities": [mapping["modality"]],
                "diagnosis": mapping["diagnosis"], "visit_id": mapping["visit_id"],
                "diagnosis_source": "Reviewed source patient map; Hwang et al. doi:10.3390/jimaging8120327"}
    return None


def inspect_dataset(root, dataset_id, disease, patient_map=None):
    """CPU headers only; one file at a time, no full volume load or GPU preprocessing."""
    import nibabel as nib
    rows, errors = [], []
    for path in sorted(Path(root).rglob("*")):
        if any(p.startswith(".") or p == "__MACOSX" for p in path.relative_to(root).parts):
            continue
        if not path.is_file() or not path.name.endswith((".nii", ".nii.gz", ".img", ".img.gz")):
            continue
        relative = path.relative_to(root).as_posix()
        try:
            image = nib.load(path)
            if len(image.shape) < 3 or min(image.shape[:3]) < 2:
                raise ValueError(f"Not a 3D MRI/mask: {image.shape}")
            identity = source_identity(relative, dataset_id, patient_map)
            if identity is not None:
                patient, role, modality = identity["patient"], identity["role"], identity["modalities"]
            elif dataset_id == "msd_task01":
                match = re.search(r"BRATS_\d+", path.name)
                patient = match[0] if match else None
                modality = ["FLAIR", "T1w", "T1ce", "T2w"] if len(image.shape) == 4 else ["mask"]
                role = "mask" if "labelsTr" in path.parts else "image"
            elif dataset_id == "openneuro_ds007345":
                match = re.search(r"sub-olf\d{2}", relative)
                patient = match[0] if match else None  # 'fu' never becomes a new patient
                role = "mask" if "segmentation" in path.parts else "image"
                modality = ["mask" if role == "mask" else ("T2w" if "_T2w" in path.name else "T1w_contrast_unspecified")]
            else:
                match = re.search(r"sub-strokecase\d+", relative)
                patient = match[0] if match else None
                role = "mask" if "_msk" in path.name else "image"
                modality = [next((m for m in ["FLAIR", "ADC", "dwi"] if m.lower() in path.name.lower()), "mask" if role == "mask" else "unknown")]
            if not patient and dataset_id != "hwang_meningioma_5945963":
                raise ValueError("Cannot establish patient/case identity from the documented layout")
            zooms = [float(v) for v in image.header.get_zooms()[:3]]
            if any(not (0 < z < 100) for z in zooms):
                raise ValueError(f"Invalid voxel spacing: {zooms}")
            header = image.file_map.get("header")
            paired_header_hash = digest(header.filename) if header and header.filename else None
            rows.append({"dataset_id": dataset_id, "patient_id": f"{dataset_id}:{patient}" if patient else None, "case_id": patient,
                         "disease": disease, "role": role, "modalities": modality, "relative_path": relative,
                         "shape": list(image.shape), "spacing_mm": zooms, "dtype": str(image.get_data_dtype()),
                         "affine": image.affine.tolist(), "bytes": path.stat().st_size,
                         "paired_header_sha256": paired_header_hash,
                         "sha256": digest(path), "diagnosis_source": identity["diagnosis_source"] if identity else "dataset-level cohort; review clinical metadata before training",
                         "diagnosis": identity["diagnosis"] if identity else disease,
                         "visit_id": identity.get("visit_id") if identity else None,
                         "source_case": identity.get("source_case") if identity else None,
                         "split_group": f"{dataset_id}:{patient}" if patient else None, "split": "unassigned"})
        except Exception as exc:
            errors.append({"path": relative, "error": f"{type(exc).__name__}: {exc}"})
    if not rows:
        errors.append({"error": "No valid 3D NIfTI files found"})
    groups = {}
    for row in rows:
        if row["patient_id"] is None:
            continue
        g = groups.setdefault(row["patient_id"], {"patient_id": row["patient_id"], "diagnosis": row["diagnosis"], "images": [], "masks": [], "ancillary": [], "split": "unassigned"})
        if g["diagnosis"] != row["diagnosis"]:
            errors.append({"error": f"Conflicting diagnoses for {row['patient_id']}"})
        g[{"image": "images", "mask": "masks"}.get(row["role"], "ancillary")].append(row["relative_path"])
    if any(not g["images"] for g in groups.values()):
        errors.append({"error": "Mask-only patient/case group found"})
    # Exact file-byte equality is a review finding, not proof of corruption or identity.
    hashes = {}
    for row in rows:
        if row["role"] == "image" and row["patient_id"]:
            hashes.setdefault(row["sha256"], []).append(row)
    duplicate_details = []
    parents = {patient: patient for patient in groups}
    def find(patient):
        while parents[patient] != patient:
            parents[patient] = parents[parents[patient]]
            patient = parents[patient]
        return patient
    for sha, matches in sorted(hashes.items()):
        patients = sorted({r["patient_id"] for r in matches})
        if len(patients) < 2:
            continue
        duplicate_details.append({"sha256": sha, "patient_ids": patients,
                                  "files": [{k: r[k] for k in ("relative_path", "case_id", "modalities")} for r in matches],
                                  "review_status": "manual_review_required",
                                  "evidence": "Exact file bytes only; patient identity is not established"})
        for patient in patients[1:]:
            parents[find(patient)] = find(patients[0])
    components = {}
    for patient in groups:
        components.setdefault(find(patient), []).append(patient)
    affected = set()
    for patients in components.values():
        patients.sort()
        linked = len(patients) > 1
        split_group = (dataset_id + ":duplicate-linked:" + hashlib.sha256("\n".join(patients).encode()).hexdigest()[:24]) if linked else patients[0]
        for patient in patients:
            groups[patient].update(split_group=split_group, manual_review_required=linked)
        if linked:
            affected.update(patients)
    for row in rows:
        if row["patient_id"] in groups:
            row["split_group"] = groups[row["patient_id"]]["split_group"]
            row["manual_review_required"] = row["patient_id"] in affected
    duplicates = [d["patient_ids"] for d in duplicate_details]
    pending = [row["relative_path"] for row in rows if row["patient_id"] is None or row["diagnosis"] is None]
    return {"files": rows, "patients": list(groups.values()), "errors": errors,
            "pending_identity_files": pending, "mapping_required": bool(pending),
            "exact_duplicate_images": duplicate_details, "manual_review_cases": sorted(affected),
            "exact_duplicate_image_groups": duplicates, "patient_count": None if any(row["patient_id"] is None for row in rows) else len(groups),
            "count_note": "MSD IDs are case groups; independent clinical identity is not proven. No cross-release deduplication is claimed."}


def validate_split_assignments(manifest, assignments):
    """Mandatory guard for any future split consumer; keys are original patient/case IDs.

    Does not assign splits or merge patients. All cases must be explicitly assigned.
    """
    patients = {p["patient_id"]: p for p in manifest["patients"]}
    if manifest["mapping_required"] or set(assignments) != set(patients):
        raise ValueError("Complete resolved case assignments required")
    groups = {}
    for patient, split in assignments.items():
        if split not in {"train", "validation", "test"}:
            raise ValueError("Unknown split")
        group = patients[patient]["split_group"]
        if group in groups and groups[group] != split:
            raise ValueError(f"Duplicate-linked cases cannot cross splits: {group}")
        groups[group] = split
    return True


def manifest_failures(spec, manifest):
    """Enumerate all hard failures, separately from duplicate review findings."""
    failures = list(manifest["errors"])
    expected = spec.get("expected_groups")
    if expected and manifest["patient_count"] is not None and manifest["patient_count"] != expected:
        failures.append({"error": f"Expected {expected} patient/case groups; found {manifest['patient_count']}"})
    if manifest["mapping_required"]:
        return failures
    for diagnosis, count in spec.get("expected_diagnosis_groups", {}).items():
        if sum(p["diagnosis"] == diagnosis for p in manifest["patients"]) != count:
            failures.append({"error": f"Unexpected group count for {diagnosis}; expected {count}"})
    if sum(r["role"] == "mask" for r in manifest["files"]) < spec.get("expected_min_masks", 0):
        failures.append({"error": "Insufficient lesion masks"})
    if spec["id"] == "isles2022":
        for patient in manifest["patients"]:
            channels = sorted((r["role"], tuple(r["modalities"])) for r in manifest["files"] if r["patient_id"] == patient["patient_id"])
            expected_channels = sorted([("image", ("FLAIR",)), ("image", ("ADC",)), ("image", ("dwi",)), ("mask", ("mask",))])
            if channels != expected_channels:
                failures.append({"patient_id": patient["patient_id"], "error": "Expected exactly one FLAIR, ADC, DWI and mask", "actual": channels})
    return failures


def record_manifest(root, spec, manifest, report):
    failures = manifest_failures(spec, manifest)
    manifest["validation_failures"] = failures
    atomic_json(root / "file_manifest.json", manifest)
    atomic_json(root / "patient_manifest.json", manifest["patients"])
    atomic_json(root / "duplicate_image_report.json", {"duplicates": manifest["exact_duplicate_images"],
                "manual_review_cases": manifest["manual_review_cases"], "action": "Preserve all files and identities; keep every connected split_group together; manual review required"})
    report.update(validation_failures=failures, file_inspection_errors=manifest["errors"],
                  exact_duplicate_image_count=len(manifest["exact_duplicate_images"]),
                  manual_review_cases=manifest["manual_review_cases"], training_ready=False,
                  patient_or_case_groups=manifest["patient_count"])
    if failures:
        raise ValueError("Manifest validation failed; see validation_failures in file_manifest.json")
    report["file_checks"] = "passed"


def revalidate_existing_isles(spec, destination):
    """Offline, read-only MRI/archive audit. Writes reports only; never downloads/extracts.

    Compare every extracted member to the publisher-verified ZIP, including full ZIP CRC.
    Missing archives/files fail closed. No assumption that header readability proves payload integrity.
    """
    root = assert_colab_drive(destination)
    if spec["id"] != "isles2022":
        raise ValueError("This offline revalidation entry point is ISLES-only")
    report = {"dataset_id": spec["id"], "status": "running", "training_ready": False,
              "mode": "existing_files_only", "archive_checks": "not_passed", "file_checks": "not_passed"}
    try:
        for asset in spec["assets"]:
            archive = checked_path(root / "downloads", asset["path"])
            algorithm, expected = asset["checksum"].split(":", 1)
            if archive.stat().st_size != asset["bytes"] or digest(archive, algorithm) != expected:
                raise ValueError("Existing archive publisher checksum/size mismatch; preserved without redownload")
            seen = set()
            with zipfile.ZipFile(archive) as container:
                for member in container.infolist():
                    mode = member.external_attr >> 16
                    if stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR) or member.flag_bits & 1:
                        raise ValueError("Unsafe ZIP member")
                    target = checked_path(root / "data", member.filename)
                    key = str(target.resolve()).casefold()
                    if key in seen:
                        raise ValueError("Duplicate ZIP member path")
                    seen.add(key)
                    if member.is_dir():
                        continue
                    h = hashlib.sha256()
                    with container.open(member) as source:
                        for chunk in iter(lambda: source.read(4 * 1024**2), b""):
                            h.update(chunk)  # Reading to EOF also verifies ZIP CRC.
                    if target.stat().st_size != member.file_size or digest(target) != h.hexdigest():
                        raise ValueError(f"Extracted file differs from verified archive: {member.filename}")
        report["archive_checks"] = "passed_publisher_checksum_and_all_member_crc_and_extracted_bytes"
        manifest = inspect_dataset(root / "data", spec["id"], spec["disease"])
        record_manifest(root, spec, manifest, report)
        report["status"] = "download_integrity_verified_review_required" if manifest["exact_duplicate_images"] else "acquired_and_inspected_not_approved_for_training"
        report["clinical_readiness"] = "manual_review_required_not_approved_for_training"
    except Exception as exc:
        report.update(status="failed", error=f"{type(exc).__name__}: {exc}")
        raise
    finally:
        atomic_json(root / "acquisition_report.json", report)
    return report


def acquire_one(spec, destination, *, download_budget_bytes, extraction_budget_bytes, patient_map_path=None, drive_available_bytes=None, download_options=None):
    root = assert_colab_drive(destination)
    if not spec.get("eligible"):
        raise RuntimeError(f"{spec['disease']}: acquisition disabled: {spec['reason']}")
    root.mkdir(parents=True, exist_ok=True)
    atomic_json(root / "source_and_license.json", spec)
    report = {"dataset_id": spec["id"], "status": "running", "training_ready": False, "assets": [], "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    try:
        known_total = sum(a.get("bytes") or 0 for a in spec["assets"])
        if known_total > download_budget_bytes:
            raise OSError("Selected assets exceed the total download budget")
        if is_drive_path(root):
            report["storage_plan"] = check_storage_plan(root, known_total,
                extraction_budget_bytes if any(a.get("archive") for a in spec["assets"]) else 0,
                drive_available_bytes=drive_available_bytes, local_root="/content")
        else:
            require_space(root, known_total)
        used = 0
        unpacked = 0
        for asset in spec["assets"]:
            result = download_verified(asset, root / "downloads", max_bytes=download_budget_bytes - used, drive_available_bytes=drive_available_bytes, **(download_options or {}))
            report["assets"].append(result)
            used += Path(result["path"]).stat().st_size
            if asset.get("archive"):
                data_destination = checked_path(root / "data", asset["extract_subdir"]) if asset.get("extract_subdir") else root / "data"
                extraction = safe_extract(result["path"], data_destination, max_unpacked_bytes=extraction_budget_bytes - unpacked, drive_available_bytes=drive_available_bytes)
                unpacked += extraction["unpacked_bytes"]
                report.setdefault("extraction", []).append(extraction)
        data_root = root / ("data" if any(a.get("archive") for a in spec["assets"]) else "downloads")
        if spec["id"] == "msd_task01":
            descriptions = list(data_root.rglob("dataset.json"))
            if len(descriptions) != 1:
                raise ValueError("Expected one MSD dataset.json to confirm source license and modalities")
            description = json.loads(descriptions[0].read_text())
            license_text = str(description.get("licence", description.get("license", ""))).lower()
            if not ("cc-by-sa" in license_text or "by-sa/4.0" in license_text):
                raise ValueError(f"Unexpected MSD license declaration: {license_text!r}; review before reuse")
        if patient_map_path:
            assert_colab_drive(Path(patient_map_path).parent)
        patient_map = load_patient_map(patient_map_path) if patient_map_path else None
        manifest = inspect_dataset(data_root, spec["id"], spec["disease"], patient_map=patient_map)
        record_manifest(root, spec, manifest, report)
        if manifest["errors"]:
            raise ValueError("NIfTI/identity inspection failed; review file_manifest.json")
        expected = spec.get("expected_groups")
        if expected and manifest["patient_count"] is not None and manifest["patient_count"] != expected:
            raise ValueError(f"Expected {expected} patient/case groups; found {manifest['patient_count']}")
        if manifest["mapping_required"]:
            # No invented patient IDs or disease/control labels for undocumented filenames.
            template = root / "patient_map_template.csv"
            if not template.exists():
                with template.open("x", newline="") as stream:
                    writer = csv.DictWriter(stream, fieldnames=["relative_path", "patient_id", "diagnosis", "modality", "role", "visit_id"])
                    writer.writeheader()
                    for row in manifest["files"]:
                        if row["relative_path"] in manifest["pending_identity_files"]:
                            writer.writerow({"relative_path": row["relative_path"], "patient_id": row["case_id"] or "",
                                             "diagnosis": row["diagnosis"] or "", "modality": row["modalities"][0] if row["role"] != "unresolved" else "",
                                             "role": row["role"] if row["role"] != "unresolved" else "", "visit_id": row["visit_id"] or ""})
            report["status"] = "download_verified_inventory_pending_patient_map"
            report["patient_or_case_groups"] = manifest["patient_count"]
            report["mapping_template"] = str(template)
            report["limitation"] = "Identity and/or diagnostic-label review is incomplete. Review source identities, repeated visits and disease/control labels; fill a separate reviewed CSV and rerun. No training is authorized."
            return report
        expected = spec.get("expected_groups")
        if expected and manifest["patient_count"] != expected:
            raise ValueError(f"Expected {expected} patient/case groups; found {manifest['patient_count']}")
        for diagnosis, count in spec.get("expected_diagnosis_groups", {}).items():
            actual = sum(p["diagnosis"] == diagnosis for p in manifest["patients"])
            if actual != count:
                raise ValueError(f"Expected {count} independent groups for {diagnosis}; found {actual}")
        minimum_masks = spec.get("expected_min_masks", 0)
        if sum(row["role"] == "mask" for row in manifest["files"]) < minimum_masks:
            raise ValueError(f"Expected at least {minimum_masks} lesion masks")
        report["archive_checks"] = "passed_download_checksums_and_safe_extraction"
        report["status"] = "download_integrity_verified_review_required" if manifest["exact_duplicate_images"] else "acquired_and_inspected_not_approved_for_training"
        report["patient_or_case_groups"] = manifest["patient_count"]
    except Exception as exc:
        report["status"] = "failed"
        report["error"] = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        atomic_json(root / "acquisition_report.json", report)
    return report


## CUDA readiness for the future Colab workflow

Acquisition, extraction, hashing and header inspection run on CPU and never load complete MRI volumes. The following tiny artificial-array check verifies CUDA, MONAI device compatibility and AMP. It is **not model training, validation, a benchmark of MRI throughput, or a checkpoint operation**. All current/peak memory and timing below describe only this self-test.

For a later separately approved training task: keep CUDA mandatory; use device-aware MONAI transforms, one-volume/lazy loading, small 3D patches and batch size 1, AMP/GradScaler and gradient accumulation where appropriate. Measure peak VRAM and tune patches below the T4 budget. Save model, optimizer, scaler, epoch and random states atomically in private Drive checkpoints and restore all of them on resume. Actual training throughput and clinical validation metrics must be measured then. No such model or loop is included or started here.


In [ ]:
if not IN_COLAB:
    raise RuntimeError("Run acquisition in Google Colab, not Codex Cloud.")
if not torch.cuda.is_available() or not REQUIRE_GPU:
    raise RuntimeError("Select a T4 CUDA runtime. CPU training fallback is forbidden.")
import time
from monai.transforms import NormalizeIntensity
properties = torch.cuda.get_device_properties(0)
VRAM_BUDGET_GIB = min(13.0, properties.total_memory / 2**30 - 2.0)
if VRAM_BUDGET_GIB <= 0:
    raise RuntimeError("Insufficient GPU memory for the future T4 workflow.")
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.no_grad():
    tiny = torch.arange(4096, dtype=torch.float32, device="cuda").reshape(1, 16, 16, 16)
    checked = NormalizeIntensity()(tiny)
    if torch.as_tensor(checked).device.type != "cuda" or not torch.isfinite(checked).all().item():
        raise RuntimeError("MONAI CUDA transform self-test failed.")
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        amp_result = torch.ones((16, 16), device="cuda") @ torch.ones((16, 16), device="cuda")
    if not torch.isfinite(amp_result).all().item():
        raise RuntimeError("CUDA AMP arithmetic self-test failed.")
torch.cuda.synchronize()
print({"gpu": properties.name, "vram_gib": round(properties.total_memory / 2**30, 2),
       "future_vram_budget_gib": VRAM_BUDGET_GIB,
       "self_test_seconds": round(time.perf_counter() - started, 4),
       "allocated_mib": round(torch.cuda.memory_allocated() / 2**20, 2),
       "peak_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 2),
       "training_started": False})
del tiny, checked, amp_result
torch.cuda.empty_cache()


## Preflight without MRI transfer

HEAD requests check all selected endpoints, current byte sizes and available storage. Drive quota uses your current confirmed free quota, never mounted Drive statvfs. Reservation includes all retained downloads plus each archived disease’s shared extraction budget, plus the greater of 5 GiB or 10% headroom. Real /content free space is checked separately (2 GiB reserve, no MRI staging). Provider/cache write failures still stop acquisition and preserve partial files; this is not a guarantee of DriveFS cache behavior. A missing Drive mount, login/error page, changed size or insufficient budget stops preflight before MRI transfer. Publisher digests remain pinned; multipart ETags are not MD5.

OASIS-2 is a separately disclosed legacy policy: authenticated HTTPS + exact byte size + full gzip CRC32, with a local SHA-256 ledger for repeat verification. It has no publisher cryptographic digest. Other sources still require their published MD5/SHA-256; this policy does not bypass a known checksum mismatch.


In [ ]:
import copy
if MENINGIOMA_SOURCE not in {"hwang91", "openneuro37"}:
    raise ValueError("Choose a reviewed meningioma source.")
selected = []
for key, enabled in SWITCHES.items():
    if enabled:
        spec = CATALOG[key]
        if key == "meningioma" and MENINGIOMA_SOURCE == "openneuro37":
            spec = spec["alternatives"]["openneuro37"]
        selected.append(copy.deepcopy(spec))
reviewed_maps = {"hwang_meningioma_5945963": MENINGIOMA_REVIEWED_MAP_CSV,
                 "oasis2_legacy": OASIS2_REVIEWED_MAP_CSV}
preflight_ok = False
if not selected:
    print("No diseases selected. No MRI files read or downloaded; IXI is untouched.")
else:
    if not IN_COLAB or not DRIVE_MOUNTED or not torch.cuda.is_available():
        raise RuntimeError("Colab CUDA runtime and mounted private Drive are required.")
    for spec in selected:
        if not spec["eligible"]:
            raise RuntimeError(f"{spec['disease']}: {spec['reason']}")
        if spec["id"] == "oasis2_legacy":
            print("OASIS-2: NO PUBLISHER CHECKSUM. HTTPS/size/gzip CRC32 + local SHA-256 ledger; clinical mapping remains required.")
    if min(DRIVE_AVAILABLE_GIB, MAX_DOWNLOAD_GIB_PER_DISEASE, MAX_UNPACKED_GIB_PER_DISEASE) <= 0:
        raise ValueError("Enter available Drive quota and positive storage budgets before download.")
    acquisition_root = assert_colab_drive(ACQUISITION_ROOT)
    acquisition_root.mkdir(parents=True, exist_ok=True)
    probes = {}
    try:
        for spec in selected:
            probes[spec["id"]] = probe_assets(spec)
        total_download = sum(asset["bytes"] for spec in selected for asset in spec["assets"])
        archive_diseases = sum(any(asset.get("archive") for asset in spec["assets"]) for spec in selected)
        storage_plan = check_storage_plan(acquisition_root, total_download,
            int(archive_diseases * MAX_UNPACKED_GIB_PER_DISEASE * 2**30),
            drive_available_bytes=int(DRIVE_AVAILABLE_GIB * 2**30), local_root="/content")
        reserve = storage_plan["reserved_bytes"]
        for spec in selected:
            if sum(asset["bytes"] for asset in spec["assets"]) > MAX_DOWNLOAD_GIB_PER_DISEASE * 2**30:
                raise OSError(f"{spec['id']} exceeds the per-disease download budget.")
        preflight_ok = True
        atomic_json(acquisition_root / "preflight_report.json", {"status": "passed", "probes": probes, "reserved_bytes": reserve, "storage_plan": storage_plan})
        print(f"Preflight passed: {total_download / 2**30:.2f} GiB download; reserve {reserve / 2**30:.2f} GiB including shared extraction budgets.")
    except Exception as exc:
        atomic_json(acquisition_root / "preflight_report.json", {"status": "failed", "probes": probes, "error": f"{type(exc).__name__}: {exc}"})
        raise


## Download, inspect and report readiness accurately

Only in Colab, stream the enabled files to Drive, verify the source's declared integrity policy, then extract with path/type/size checks. PROTEAS course archives have separate extraction directories; a/b courses retain a shared patient ID. CT, radiation plans and brain-tissue labels are ancillary, not MRI sequences or metastasis masks. Raw DICOM is retained without conversion.

Headers are inspected one file at a time without loading full MRI volumes. NIfTI-1 paired IMG/HDR is supported, including a separate header hash. Source IDs group OASIS visits; undocumented meningioma identities remain unknown until a reviewed mapping is supplied. No incomplete identity/diagnosis mapping is reported as training ready.

Failures are persisted per disease while other independent selections continue. Different existing files are preserved. Interrupted extraction files require inspection; existing reports are retained on rerun. No code in this notebook trains or loads a model.


In [ ]:
results = []
if selected and not preflight_ok:
    raise RuntimeError("Rerun and pass preflight before transferring any MRI data.")
for spec in selected:
    try:
        result = acquire_one(spec, str(Path(ACQUISITION_ROOT) / spec["id"]),
                             download_budget_bytes=int(MAX_DOWNLOAD_GIB_PER_DISEASE * 2**30),
                             extraction_budget_bytes=int(MAX_UNPACKED_GIB_PER_DISEASE * 2**30),
                             patient_map_path=reviewed_maps.get(spec["id"]) or None,
                             drive_available_bytes=int(DRIVE_AVAILABLE_GIB * 2**30),
                             download_options=DOWNLOAD_OPTIONS if spec["id"] == "hwang_meningioma_5945963" else None)
        results.append(result)
        print(spec["id"], result["status"], result.get("patient_or_case_groups"))
        if result.get("mapping_template"):
            print("Private mapping template:", result["mapping_template"])
            print(result["limitation"])
    except Exception as exc:
        results.append({"dataset_id": spec["id"], "status": "failed", "error": f"{type(exc).__name__}: {exc}"})
        print(spec["id"], "FAILED:", type(exc).__name__, str(exc))
if selected:
    atomic_json(Path(ACQUISITION_ROOT) / "acquisition_summary.json", results)
    if any(r["status"] == "failed" for r in results):
        raise RuntimeError("At least one acquisition failed. Review private Drive reports; no training is authorized.")
    pending = [r["dataset_id"] for r in results if r["status"] == "download_verified_inventory_pending_patient_map"]
    if pending:
        print("Downloaded/inventoried, but patient/clinical review still pending:", pending)
print("Acquisition stage ended. No models trained; verification and user approval remain prerequisites for training.")


## Revalidate existing ISLES without downloading or extracting

For the reported duplicate FLAIR in sub-strokecase0090/sub-strokecase0169, keep every download switch False. Run setup/dependencies, mount Drive, configuration, embedded catalog and acquisition-engine cells; skip preflight and execute-acquisition. Enable only the switch below and run this cell. Use the same PROJECT_ROOT as the original acquisition. No GPU computation is needed for this CPU integrity audit.

This requires the retained downloads/ISLES-2022.zip and data/ directory. It verifies pinned publisher MD5/size, full ZIP-member CRC, every extracted member's bytes, MRI headers/hashes, 250 case groups, and exactly one FLAIR/ADC/DWI/mask per case. It writes reports only; missing or altered files stop the audit without downloading, extracting, deleting or replacing MRI. Expect substantial Drive read I/O.

Read acquisition_report.json, file_manifest.json, patient_manifest.json and duplicate_image_report.json under the ISLES dataset directory. validation_failures must be empty and archive_checks/file_checks must pass before integrity success is reported. Exact duplicate SHA256 findings are separate from file inspection errors. Both original cases and masks remain; all files of duplicate-linked cases share a conservative split_group, including transitive links. This does not assert a shared patient. Future split code must call validate_split_assignments(manifest, assignments) using original namespaced case IDs before exporting any splits. Clinical review and user approval remain pending even after integrity verification.


In [ ]:
REVALIDATE_EXISTING_ISLES = False  # @param {type:"boolean"}
if REVALIDATE_EXISTING_ISLES:
    existing_isles_root = Path(ACQUISITION_ROOT) / CATALOG["post_stroke"]["id"]
    isles_report = revalidate_existing_isles(CATALOG["post_stroke"], existing_isles_root)
    print(json.dumps(isles_report, indent=2))
else:
    print("Existing ISLES audit disabled; no files read.")


## Stop before training

Review every private source/license record, acquisition report and file/patient manifest. For OASIS-2, finish the clinical-group join and exclude all converter visits from the binary AD/control cohort. For the larger meningioma source, resolve longitudinal patient IDs and source-specific control labels; the paper/supplement disagree about the control count. Missing masks or labels must not become healthy labels.

The original 37-patient meningioma choice remains an alternative; do not download both just to inflate sample counts. PROTEAS versions overlap and are never independent cohorts. Keep every patient's repeated visits, channels and slices in one split. Review exact/near duplicates, mask geometry and source provenance before creating splits.

IXI T2 controls differ in scanner, modality, age and site from these disease cohorts; this notebook does not read or combine IXI. No GPU MRI preprocessing, validation experiment or training has been performed. Those belong to a later, approved Colab T4 task with CUDA, memory-efficient patches, AMP and complete Drive checkpoint/resume state.
